# LNMESCC CE-CT Multimodal Analysis

This is the cleaned public version of the final analysis notebook accompanying the manuscript. It contains only analyses cited in the main text or Supplementary Tables S2-S3 and Supplementary Figure S2.

The repository does not contain individual-level clinical data, imaging data, feature matrices, patient identifiers, or patient-level predictions. Read `../README.md` and `../data/README.md` before running this notebook. Run all cells from top to bottom in a clean Python 3.9 environment.


## 1. Development pipeline and training-only margin selection

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns  # 保留不影响运行（你原代码里有）
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from scipy import stats
from sklearn.feature_selection import VarianceThreshold

# ================= Public repository configuration =================
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
elif not (REPO_ROOT / "notebooks").exists() and (REPO_ROOT.parent / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = Path(os.environ.get("LNMESCC_DATA_DIR", str(REPO_ROOT / "data")))
OUTPUT_DIR = Path(os.environ.get("LNMESCC_OUTPUT_DIR", str(REPO_ROOT / "results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_TRAIN_CLIN = str(DATA_DIR / "Clean_Train_Set.xlsx")
FILE_VAL_CLIN = str(DATA_DIR / "Clean_Val_Set.xlsx")
FILE_RAD_DL = str(DATA_DIR / "Rad+DL_Cleaned_Features.xlsx")
DIR_HABITAT = str(DATA_DIR / "Habitat_Raw")
BASE_DIR = str(OUTPUT_DIR)
SEED = 42
PCC_TH = 0.8

# Patient-level exports are disabled in the public release by default.
EXPORT_PATIENT_LEVEL_PREDICTIONS = False
# =====================================================================


class FusionPipeline:
    def __init__(self):
        self.train_data = None
        self.val_data = None
        self.best_margin = None
        self.best_rad_feats = []
        self.selected_clin_vars = []

    # --------------------- Data Loading ---------------------
    def _pivot_rad_dl(self, df_raw):
        print(f"   ⚡ 正在透视转换 Rad+DL 表...")
        id_col = df_raw.columns[0]
        df_raw[id_col] = df_raw[id_col].astype(str)
        exclude_cols = [id_col, 'MaskType', 'Group', 'Target', 'LNM', 'N0']
        feature_cols = [c for c in df_raw.columns if c not in exclude_cols]

        df_pivot = df_raw.pivot(index=id_col, columns='MaskType', values=feature_cols)
        new_cols = [f"{mask_type}_{feat_name}" for feat_name, mask_type in df_pivot.columns]
        df_pivot.columns = new_cols
        df_pivot.index.name = 'ID'
        return df_pivot

    def _load_habitat_features(self):
        print("   🦜 正在读取生境文件...")
        regions, labels, hab_frames = ['t', 'p2', 'p3', 'p4'], [1, 2, 3, 4], []
        for r in regions:
            for l in labels:
                path = os.path.join(DIR_HABITAT, f"{r}_Habitat_{l}.xlsx")
                if not os.path.exists(path):
                    continue
                df = pd.read_excel(path)
                id_col = df.columns[0]
                df[id_col] = df[id_col].astype(str)
                df = df.set_index(id_col).add_prefix(f"Hab_{r}_{l}_")
                hab_frames.append(df)
        return pd.concat(hab_frames, axis=1).fillna(0) if hab_frames else pd.DataFrame()

    def load_data(self):
        print("📥 Step 1: Loading & Merging Data...")
        df_tr_clin = pd.read_excel(FILE_TRAIN_CLIN).set_index('ID')
        df_va_clin = pd.read_excel(FILE_VAL_CLIN).set_index('ID')
        df_tr_clin.index = df_tr_clin.index.astype(str)
        df_va_clin.index = df_va_clin.index.astype(str)

        df_rad_dl = self._pivot_rad_dl(pd.read_excel(FILE_RAD_DL))
        df_hab = self._load_habitat_features()

        self.train_data = df_tr_clin.join([df_rad_dl, df_hab], how='inner')
        self.val_data = df_va_clin.join([df_rad_dl, df_hab], how='inner')
        print(f"   📊 最终数据量: Train={len(self.train_data)}, Val={len(self.val_data)}")

        if 'Target' not in self.train_data.columns or 'Target' not in self.val_data.columns:
            raise RuntimeError("Missing 'Target' column after merging. Check clinical input files.")

    # --------------------- Feature Utilities ---------------------
    def _pcc_reduction(self, cols, threshold=PCC_TH):
        """PCC 去冗余：保留与 y 相关性更强的特征；threshold=0.8（按需求）"""
        if len(cols) < 2:
            return cols

        X = self.train_data[cols].apply(pd.to_numeric, errors='coerce')
        y = self.train_data['Target']

        # 去掉零方差
        selector = VarianceThreshold(threshold=0)
        selector.fit(X.fillna(0))
        cols_var = X.columns[selector.get_support()].tolist()
        if len(cols_var) == 0:
            return []
        if len(cols_var) < len(cols):
            X = X[cols_var]

        # 相关矩阵
        corrs_y = X.corrwith(y).abs()
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        to_drop = set()
        high_corr_pairs = upper.stack()
        high_corr_pairs = high_corr_pairs[high_corr_pairs > threshold]

        for (f1, f2), _ in high_corr_pairs.items():
            if f1 in to_drop or f2 in to_drop:
                continue
            if corrs_y.get(f1, 0) >= corrs_y.get(f2, 0):
                to_drop.add(f2)
            else:
                to_drop.add(f1)

        kept = [c for c in cols_var if c not in to_drop]
        return kept

    # ====== (改动 1) margin 决策严格折内流程：新增 df 版本 PCC/RF-IFS + 外层评估 ======
    def _pcc_reduction_on_df(self, df, cols, threshold=PCC_TH):
        """PCC 去冗余（fold内版）：只用 df(外层训练子集) 计算 PCC 与 y 相关性，避免信息泄漏"""
        if len(cols) < 2:
            return cols

        X = df[cols].apply(pd.to_numeric, errors='coerce')
        y = df['Target']

        selector = VarianceThreshold(threshold=0)
        selector.fit(X.fillna(0))
        cols_var = X.columns[selector.get_support()].tolist()
        if len(cols_var) == 0:
            return []
        if len(cols_var) < len(cols):
            X = X[cols_var]

        corrs_y = X.corrwith(y).abs()
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

        to_drop = set()
        high_corr_pairs = upper.stack()
        high_corr_pairs = high_corr_pairs[high_corr_pairs > threshold]

        for (f1, f2), _ in high_corr_pairs.items():
            if f1 in to_drop or f2 in to_drop:
                continue
            if corrs_y.get(f1, 0) >= corrs_y.get(f2, 0):
                to_drop.add(f2)
            else:
                to_drop.add(f1)

        kept = [c for c in cols_var if c not in to_drop]
        return kept

    def _rf_ifs_select(self, cols, name):
        """RF-IFS：用RF重要性排序，逐步增加k，用 Train 内部 CV AUC 找最佳k"""
        if len(cols) < 2:
            return cols
        print(f"      🌲 Running RF-IFS for [{name}] (Input: {len(cols)})...")

        X_tr = self.train_data[cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
        y_tr = self.train_data['Target'].values

        rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        importances = rf.feature_importances_

        indices = np.argsort(importances)[::-1]
        sorted_cols = [cols[i] for i in indices]

        max_k = min(len(sorted_cols), 30)
        best_auc = -1
        best_k = 1

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        print(f"      Running IFS steps (Max {max_k}): ", end="")
        for k in range(1, max_k + 1):
            if k % 10 == 0:
                print(".", end="")
            X_subset = X_tr[:, indices[:k]]
            rf_eval = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)
            mean_auc = cross_val_score(
                rf_eval, X_subset, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1
            ).mean()

            if mean_auc > best_auc:
                best_auc = mean_auc
                best_k = k

        print(f" Done! Best k={best_k}")
        final_features = sorted_cols[:best_k]
        print(f"      ✅ RF-IFS Selected Top {len(final_features)} features (CV AUC={best_auc:.3f})")
        return final_features

    def _rf_ifs_select_on_df(self, df, cols, name):
        """RF-IFS（fold内版）：只用 df(外层训练子集) 做排序+IFS，避免信息泄漏"""
        if len(cols) < 2:
            return cols
        print(f"      🌲 Running RF-IFS for [{name}] (Input: {len(cols)})...")

        X_tr = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
        y_tr = df['Target'].values

        rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        importances = rf.feature_importances_

        indices = np.argsort(importances)[::-1]
        sorted_cols = [cols[i] for i in indices]

        max_k = min(len(sorted_cols), 30)
        best_auc = -1
        best_k = 1

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

        print(f"      Running IFS steps (Max {max_k}): ", end="")
        for k in range(1, max_k + 1):
            if k % 10 == 0:
                print(".", end="")
            X_subset = X_tr[:, indices[:k]]
            rf_eval = RandomForestClassifier(n_estimators=50, random_state=SEED, n_jobs=-1)
            mean_auc = cross_val_score(
                rf_eval, X_subset, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1
            ).mean()

            if mean_auc > best_auc:
                best_auc = mean_auc
                best_k = k

        print(f" Done! Best k={best_k}")
        final_features = sorted_cols[:best_k]
        print(f"      ✅ RF-IFS Selected Top {len(final_features)} features (CV AUC={best_auc:.3f})")
        return final_features

    def _outer_fold_auc_full_pipeline(self, feats_all, name, n_splits=5):
        """
        外层 CV：每个 fold 内执行 PCC -> RF-IFS -> LASSO(CV选C)，再在该 fold 测试子集评估 AUC。
        这是“全流程嵌套”（特征选择也折内完成），用于 margin 决策。
        """
        if len(feats_all) == 0:
            return 0.0

        df_all = self.train_data
        y_all = df_all['Target'].values
        outer = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        hard_Cs = np.logspace(-4, -0.3, 20)

        aucs = []
        fold_id = 0

        for tr_idx, te_idx in outer.split(np.zeros(len(y_all)), y_all):
            fold_id += 1
            df_tr = df_all.iloc[tr_idx].copy()
            df_te = df_all.iloc[te_idx].copy()

            print(f"   🔁 [{name}] Outer fold {fold_id}/{n_splits}...")

            # --- Fold内 PCC ---
            feats_pcc = self._pcc_reduction_on_df(df_tr, feats_all, PCC_TH)
            if len(feats_pcc) == 0:
                aucs.append(0.5)
                print(f"      ⚠️ [{name}] fold {fold_id}: PCC left 0 feats -> AUC=0.500")
                continue

            # --- Fold内 RF-IFS ---
            feats_rf = self._rf_ifs_select_on_df(df_tr, feats_pcc, f"{name}_fold{fold_id}")
            if len(feats_rf) == 0:
                aucs.append(0.5)
                print(f"      ⚠️ [{name}] fold {fold_id}: RF-IFS left 0 feats -> AUC=0.500")
                continue

            # --- Fold内 LASSO(CV选C)，Fold外层测试评估 ---
            X_tr = df_tr[feats_rf].apply(pd.to_numeric, errors='coerce').values
            y_tr = df_tr['Target'].values
            X_te = df_te[feats_rf].apply(pd.to_numeric, errors='coerce').values
            y_te = df_te['Target'].values

            imp = SimpleImputer(strategy='mean')
            X_tr = imp.fit_transform(X_tr)
            X_te = imp.transform(X_te)

            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)

            mdl = LogisticRegressionCV(
                Cs=hard_Cs,
                cv=5,
                penalty='l1',
                solver='liblinear',
                scoring='roc_auc',
                max_iter=10000,
                random_state=SEED,
                n_jobs=-1
            )
            mdl.fit(X_tr, y_tr)
            p = mdl.predict_proba(X_te)[:, 1]
            auc = roc_auc_score(y_te, p)
            aucs.append(float(auc))
            print(f"      ✅ [{name}] fold {fold_id}: AUC={auc:.3f} | feats={len(feats_rf)}")

        mean_auc = float(np.mean(aucs)) if len(aucs) else 0.0
        print(f"      🧾 [{name}] Full-pipeline Outer-CV mean AUC = {mean_auc:.3f}")
        return mean_auc
    # ====== (改动 1结束) ======

    # --------------------- Leak-free LASSO scoring ---------------------
    def _nested_cv_auc_lasso(self, df, cols, name, n_splits=5):
        """
        ✅ Train-only 嵌套CV AUC：用于 margin 决策，不触碰 Val
        外层fold评估，内层 LogisticRegressionCV 选 C
        """
        if len(cols) == 0:
            return 0.0

        X = df[cols].apply(pd.to_numeric, errors='coerce').values
        y = df['Target'].values

        hard_Cs = np.logspace(-4, -0.3, 20)
        outer = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        aucs = []

        for tr_idx, te_idx in outer.split(X, y):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            imp = SimpleImputer(strategy='mean')
            X_tr = imp.fit_transform(X_tr)
            X_te = imp.transform(X_te)

            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)

            mdl = LogisticRegressionCV(
                Cs=hard_Cs,
                cv=5,
                penalty='l1',
                solver='liblinear',
                scoring='roc_auc',
                max_iter=10000,
                random_state=SEED,
                n_jobs=-1
            )
            mdl.fit(X_tr, y_tr)
            p = mdl.predict_proba(X_te)[:, 1]
            aucs.append(roc_auc_score(y_te, p))

        mean_auc = float(np.mean(aucs))
        print(f"      🔁 [{name}] Nested-CV AUC = {mean_auc:.3f} (Train only)")
        return mean_auc

    def _fit_lasso_scores(self, train_df, val_df, cols, name, n_splits=5):
        """
        ✅ 生成签名分数（无信息泄漏）：
        - Train: OOF 预测(更真实)
        - Val: 用全Train拟合后的模型预测（Val仅用于最终评估/监控，不用于选择）
        返回：(train_oof_score, val_score, selected_feature_names)
        """
        if len(cols) == 0:
            return np.zeros(len(train_df)), np.zeros(len(val_df)), []

        X_tr_all = train_df[cols].apply(pd.to_numeric, errors='coerce').values
        y_tr_all = train_df['Target'].values
        X_va_all = val_df[cols].apply(pd.to_numeric, errors='coerce').values

        hard_Cs = np.logspace(-4, -0.3, 20)

        # --- Train OOF ---
        oof = np.zeros(len(train_df))
        outer = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

        for tr_idx, te_idx in outer.split(X_tr_all, y_tr_all):
            X_tr, X_te = X_tr_all[tr_idx], X_tr_all[te_idx]
            y_tr = y_tr_all[tr_idx]

            imp = SimpleImputer(strategy='mean')
            X_tr = imp.fit_transform(X_tr)
            X_te = imp.transform(X_te)

            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_te = sc.transform(X_te)

            mdl = LogisticRegressionCV(
                Cs=hard_Cs,
                cv=5,
                penalty='l1',
                solver='liblinear',
                scoring='roc_auc',
                max_iter=10000,
                random_state=SEED,
                n_jobs=-1
            )
            mdl.fit(X_tr, y_tr)
            oof[te_idx] = mdl.predict_proba(X_te)[:, 1]

        # --- Refit on full Train, predict Val ---
        imp = SimpleImputer(strategy='mean')
        X_tr_imp = imp.fit_transform(X_tr_all)
        X_va_imp = imp.transform(X_va_all)

        sc = StandardScaler()
        X_tr_sc = sc.fit_transform(X_tr_imp)
        X_va_sc = sc.transform(X_va_imp)

        final_mdl = LogisticRegressionCV(
            Cs=hard_Cs,
            cv=5,
            penalty='l1',
            solver='liblinear',
            scoring='roc_auc',
            max_iter=10000,
            random_state=SEED,
            n_jobs=-1
        )
        final_mdl.fit(X_tr_sc, y_tr_all)
        p_va = final_mdl.predict_proba(X_va_sc)[:, 1]

        # --- Extract selected features ---
        coefs = final_mdl.coef_.flatten()
        idx = np.where(coefs != 0)[0]
        selected_feats = [cols[i] for i in idx]

        print(f"\n   ⚙️ [{name}] LASSO Retained {len(selected_feats)} feats (Train-only CV for C)")
        if len(selected_feats) > 0:
            top = sorted([(cols[i], coefs[i]) for i in idx], key=lambda x: abs(x[1]), reverse=True)[:5]
            print("      📝 Top Features:")
            for fn, fv in top:
                print(f"         * {fn[:40]:<40} : {fv:.4f}")

        # 仅监控（不用于选择/调参）
        auc_oof = roc_auc_score(y_tr_all, oof)
        auc_va = roc_auc_score(val_df['Target'].values, p_va)
        print(f"      📈 Monitor: Train(OOF) AUC={auc_oof:.3f} | Val AUC={auc_va:.3f} | Gap={auc_oof-auc_va:.3f}")

        return oof, p_va, selected_feats

    # --------------------- Step 2: Margin Tournament ---------------------
    def run_margin_tournament(self):
        print("\n🏆 Step 2: Margin Tournament (Radiomics Only, FULL Pipeline Outer-CV, no leakage)...")
        cols = self.train_data.columns
        rad_t = [c for c in cols if c.startswith('t_Rad_')]

        margins = {
            '2mm': rad_t + [c for c in cols if c.startswith('p2_Rad_')],
            '3mm': rad_t + [c for c in cols if c.startswith('p3_Rad_')],
            '4mm': rad_t + [c for c in cols if c.startswith('p4_Rad_')]
        }

        best_auc = -1
        best_m = None

        for m, feats in margins.items():
            if not feats:
                continue

            # ====== (改动 1) 严格折内：外层 fold 内做 PCC -> RF-IFS -> LASSO(CV选C)，再在该 fold 测试集评估 ======
            auc_cv = self._outer_fold_auc_full_pipeline(feats, f"Margin_{m}", n_splits=5)
            # ====== (改动 1结束) ======

            if auc_cv > best_auc:
                best_auc = auc_cv
                best_m = m

        self.best_margin = best_m
        if self.best_margin is None:
            raise RuntimeError("No valid margin found. Check Radiomics feature prefixes and merged columns.")
        print(f"🎉 Winner is: {self.best_margin} (Full-pipeline Outer-CV mean AUC={best_auc:.4f})")

    # --------------------- Step 3: Build Signatures ---------------------
    def build_signatures(self):
        print("\n🏗️ Step 3: Building Signatures (PCC=0.8 -> RF-IFS -> LASSO, no Val leakage)...")
        best_p = f"p{self.best_margin[0]}"
        cols = self.train_data.columns

        # === 1. Rad ===
        print("   🔹 Processing Radiomics...")
        feats_rad = [c for c in cols if c.startswith('t_Rad_') or c.startswith(f'{best_p}_Rad_')]
        feats_rad = self._pcc_reduction(feats_rad, PCC_TH)
        feats_rad = self._rf_ifs_select(feats_rad, "Rad")
        r_tr, r_va, self.best_rad_feats = self._fit_lasso_scores(self.train_data, self.val_data, feats_rad, "Rad-score")

        # === 2. DL ===
        print("   🔹 Processing DL...")
        feats_dl = [c for c in cols if c.startswith('t_DL_') or c.startswith(f'{best_p}_DL_')]
        feats_dl = self._pcc_reduction(feats_dl, PCC_TH)
        feats_dl = self._rf_ifs_select(feats_dl, "DL")
        d_tr, d_va, _ = self._fit_lasso_scores(self.train_data, self.val_data, feats_dl, "DL-score")

        # === 3. Habitat ===
        print("   🔹 Processing Habitat...")
        feats_hab = [c for c in cols if c.startswith('Hab_t_') or c.startswith(f'Hab_{best_p}_')]
        feats_hab = self._pcc_reduction(feats_hab, PCC_TH)
        feats_hab = self._rf_ifs_select(feats_hab, "Hab")
        h_tr, h_va, _ = self._fit_lasso_scores(self.train_data, self.val_data, feats_hab, "Hab-score")

        self.train_data['Rad_score'] = r_tr
        self.val_data['Rad_score'] = r_va
        self.train_data['DL_score'] = d_tr
        self.val_data['DL_score'] = d_va
        self.train_data['Hab_score'] = h_tr
        self.val_data['Hab_score'] = h_va

    # --------------------- Step 3.5: Clinical Screening ---------------------
    def _bh_fdr(self, pval_dict, q=0.05):
        """
        Benjamini–Hochberg FDR 控制（返回通过的变量名列表与 cutoff）
        """
        s = pd.Series(pval_dict, dtype=float).dropna()
        m = int(s.shape[0])
        if m == 0:
            return [], None

        s_sorted = s.sort_values()
        ranks = np.arange(1, m + 1, dtype=float)
        thresh = (ranks / m) * q
        passed = s_sorted.values <= thresh

        if not np.any(passed):
            return [], None

        max_i = int(np.where(passed)[0].max())
        cutoff = float(s_sorted.iloc[max_i])
        selected = s.index[s.values <= cutoff].tolist()
        return selected, cutoff

    def step_select_clinical_features(self):
        print("\n🩺 Step 3.5: Screening Clinical Features (Univariate + FDR BH q=0.05)...")
        # ✅ 改为精确排除（避免 substring 误伤）
        exclude_exact = set(['Target', 'Group', 'Rad_score', 'DL_score', 'Hab_score', 'LNM', 'N0'])

        # 排除影像/生境前缀
        candidates = []
        for c in self.train_data.columns:
            if c in exclude_exact:
                continue
            if c.startswith(('t_', 'p2_', 'p3_', 'p4_', 'Hab_')):
                continue
            candidates.append(c)

        y = self.train_data['Target']

        # ====== (改动 2) 先计算所有候选变量 p 值，再做 FDR(BH) 筛选 ======
        pvals = {}

        for col in candidates:
            try:
                x_raw = self.train_data[col]
                if x_raw.nunique(dropna=True) <= 1:
                    continue

                # 稳健填充：mode为空时用0
                m = x_raw.mode(dropna=True)
                fillv = m.iloc[0] if len(m) else 0
                x = x_raw.fillna(fillv)

                # 判定离散/连续：沿用你的逻辑（<=5 当类别）
                p = 1.0
                if x.nunique() <= 5:
                    ct = pd.crosstab(x, y)
                    if ct.shape[0] > 1 and ct.shape[1] > 1:
                        _, p, _, _ = stats.chi2_contingency(ct)
                else:
                    g0, g1 = x[y == 0], x[y == 1]
                    if len(g0) > 0 and len(g1) > 0:
                        _, p = stats.mannwhitneyu(g0, g1)

                pvals[col] = float(p)
            except:
                continue

        selected_fdr, cutoff = self._bh_fdr(pvals, q=0.05)

        if cutoff is None:
            print("   ⚠️ FDR(BH) 无变量通过（q=0.05）。将触发 fallback 逻辑。")
        else:
            print(f"   ✅ FDR(BH) cutoff p = {cutoff:.4g} | Passed = {len(selected_fdr)}")

        self.selected_clin_vars = selected_fdr if selected_fdr else (['Age'] if 'Age' in candidates else [])
        print(f"   🎯 Selected: {self.selected_clin_vars}")
        # ====== (改动 2结束) ======

    # --------------------- Step 4: Fusion (scaled LR) ---------------------
    def run_fusion(self):
        print("\n⚗️ Step 4: Final Fusion (LR + StandardScaler)...")
        clin = self.selected_clin_vars

        models = {
            'Model 1 (Clin Only)': clin,
            'Model 2 (+Rad)': clin + ['Rad_score'],
            'Model 3 (+DL)': clin + ['Rad_score', 'DL_score'],
            'Model 4 (+Hab)': clin + ['Rad_score', 'DL_score', 'Hab_score']
        }

        plt.figure(figsize=(9, 7))
        colors = ['grey', 'blue', 'orange', 'red']

        for (name, feats), color in zip(models.items(), colors):
            # 特征存在性检查
            missing = [f for f in feats if f not in self.train_data.columns]
            if missing:
                raise RuntimeError(f"Missing features in train_data for {name}: {missing}")

            X_tr = self.train_data[feats].apply(pd.to_numeric, errors='coerce').values
            X_va = self.val_data[feats].apply(pd.to_numeric, errors='coerce').values
            y_tr = self.train_data['Target'].values
            y_va = self.val_data['Target'].values

            imp = SimpleImputer(strategy='mean')
            X_tr = imp.fit_transform(X_tr)
            X_va = imp.transform(X_va)

            # ✅ 融合阶段标准化
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_va = sc.transform(X_va)

            lr = LogisticRegression(random_state=SEED, max_iter=5000)
            lr.fit(X_tr, y_tr)

            p_tr = lr.predict_proba(X_tr)[:, 1]
            p_va = lr.predict_proba(X_va)[:, 1]
            auc_tr = roc_auc_score(y_tr, p_tr)
            auc_va = roc_auc_score(y_va, p_va)
            fpr, tpr, _ = roc_curve(y_va, p_va)

            print(f"   📊 {name}: Train AUC={auc_tr:.3f} | Val AUC={auc_va:.3f} | Gap={auc_tr - auc_va:.3f}")
            plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc_va:.3f})')

        plt.plot([0, 1], [0, 1], 'k--')
        plt.legend(loc='lower right')
        plt.title('Final Multi-modal Fusion ROC')
        plt.show()


if __name__ == "__main__":
    pipe = FusionPipeline()
    pipe.load_data()
    pipe.run_margin_tournament()
    pipe.build_signatures()
    pipe.step_select_clinical_features()


## 2. Shared data helpers

In [ ]:
# ===== 生成 pred_store（Train OOF + Val），用于后续所有绘图/指标表 =====
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def _coerce_numeric(df: pd.DataFrame, feats: list) -> pd.DataFrame:
    """
    将特征列尽量转为数值型；若遇到 T stage 这类 'T1/T2/T3' 字符串，自动映射到 1/2/3。
    其他无法转换的 object 列会直接报错（避免静默把重要变量变成 NaN）。
    """
    out = pd.DataFrame(index=df.index)
    map_t = {'T1': 1, 'T2': 2, 'T3': 3, 't1': 1, 't2': 2, 't3': 3,
             '1': 1, '2': 2, '3': 3}

    for c in feats:
        s = df[c]
        if s.dtype == object:
            s_str = s.astype(str).str.strip()
            s_num1 = pd.to_numeric(s_str, errors='coerce')      # 优先直接转数字
            s_num2 = s_str.map(map_t)                           # 再尝试 T1/T2/T3 映射
            s_num = s_num1.copy()
            s_num[s_num.isna()] = s_num2[s_num.isna()]

            if s_num.isna().all():
                raise ValueError(
                    f"特征列 '{c}' 为 object 且无法转换为数值。"
                    f"请先在数据中把它编码为数值/哑变量再建模。"
                )
            out[c] = s_num
        else:
            out[c] = pd.to_numeric(s, errors='coerce')

    return out



## 3. Locked-margin internal-test report

In [ ]:
# =========================
# Val 集独立检验：AUC汇总 + DeLong/Bootstrap (4mm vs 2mm/3mm)
# 输出到results文件夹
# =========================

import os
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from scipy.stats import norm

# ---------- 你results路径（若你已在主代码里设置 BASE_DIR，也可以直接沿用） ----------
BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

# ---------- 如果 pipe 不存在，就创建并加载 ----------
if "pipe" not in globals():
    pipe = FusionPipeline()
    pipe.load_data()

# ---------- DeLong: correlated AUC test ----------
def _compute_midrank(x):
    x = np.asarray(x)
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1.0
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

def _fast_delong(preds_sorted_transposed, label_1_count):
    m = int(label_1_count)
    n = preds_sorted_transposed.shape[1] - m
    k = preds_sorted_transposed.shape[0]

    pos = preds_sorted_transposed[:, :m]
    neg = preds_sorted_transposed[:, m:]

    tx = np.empty((k, m), dtype=float)
    ty = np.empty((k, n), dtype=float)
    tz = np.empty((k, m + n), dtype=float)

    for r in range(k):
        tx[r, :] = _compute_midrank(pos[r, :])
        ty[r, :] = _compute_midrank(neg[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])

    aucs = (tz[:, :m].sum(axis=1) - m * (m + 1) / 2.0) / (m * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m

    sx = np.cov(v01)
    sy = np.cov(v10)
    delong_cov = sx / m + sy / n
    return aucs, delong_cov

def delong_pvalue(y_true, scores_a, scores_b):
    """
    DeLong test p-value for AUC difference between two correlated score sets.
    Returns: auc_a, auc_b, p_value
    """
    y_true = np.asarray(y_true).astype(int).ravel()
    scores_a = np.asarray(scores_a).astype(float).ravel()
    scores_b = np.asarray(scores_b).astype(float).ravel()

    order = np.argsort(-y_true)  # positives first
    y_sorted = y_true[order]
    sA = scores_a[order]
    sB = scores_b[order]
    m = int(y_sorted.sum())
    if m == 0 or m == len(y_sorted):
        raise ValueError("y_true must contain both classes for DeLong test.")

    preds = np.vstack([sA, sB])
    aucs, cov = _fast_delong(preds, m)

    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2.0 * cov[0, 1]
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), 1.0

    z = diff / np.sqrt(var)
    p = 2.0 * (1.0 - norm.cdf(abs(z)))
    return float(aucs[0]), float(aucs[1]), float(p)

# ---------- paired bootstrap ----------
def bootstrap_auc_ci(y_true, scores, n_boot=5000, seed=42):
    """
    Bootstrap AUC distribution on one model.
    Returns: auc_point, auc_boot_mean, auc_boot_sd, ci_low, ci_high
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    s = np.asarray(scores).astype(float)
    n = len(y_true)

    auc_point = float(roc_auc_score(y_true, s))
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb = y_true[idx]
        if yb.min() == yb.max():
            continue
        aucs.append(roc_auc_score(yb, s[idx]))
    aucs = np.asarray(aucs, dtype=float)

    return (auc_point,
            float(aucs.mean()),
            float(aucs.std(ddof=1)),
            float(np.percentile(aucs, 2.5)),
            float(np.percentile(aucs, 97.5)))

def bootstrap_auc_diff(y_true, scores_a, scores_b, n_boot=5000, seed=42):
    """
    Paired bootstrap on AUC difference (A - B) using same resampled indices.
    Returns: diff_mean, ci_low, ci_high, p_two_sided
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    a = np.asarray(scores_a).astype(float)
    b = np.asarray(scores_b).astype(float)
    n = len(y_true)

    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb = y_true[idx]
        if yb.min() == yb.max():
            continue
        diffs.append(roc_auc_score(yb, a[idx]) - roc_auc_score(yb, b[idx]))
    diffs = np.asarray(diffs, dtype=float)

    diff_mean = float(diffs.mean())
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5]).astype(float)

    p = 2.0 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    p = float(min(p, 1.0))
    return diff_mean, float(ci_low), float(ci_high), p

# ---------- 训练(仅Train) -> 预测Val 的函数 ----------
def train_pipeline_predict_val(pipe, margin="4mm", pcc_th=0.8, seed=42, max_k=30):
    """
    Train-only: PCC -> RF-IFS -> LASSO(LogisticRegressionCV) on Train,
    then predict Val. Return predictions + AUC + feature counts + selected feats.
    """
    df_tr = pipe.train_data
    df_va = pipe.val_data

    cols = df_tr.columns
    rad_t = [c for c in cols if c.startswith("t_Rad_")]
    if margin == "2mm":
        feats_all = rad_t + [c for c in cols if c.startswith("p2_Rad_")]
    elif margin == "3mm":
        feats_all = rad_t + [c for c in cols if c.startswith("p3_Rad_")]
    elif margin == "4mm":
        feats_all = rad_t + [c for c in cols if c.startswith("p4_Rad_")]
    else:
        raise ValueError("margin must be one of: '2mm','3mm','4mm'")

    if len(feats_all) == 0:
        raise RuntimeError(f"No radiomics features found for margin={margin}. Check prefixes t_Rad_/p*_Rad_")

    # --- PCC on full Train ---
    feats_pcc = pipe._pcc_reduction(feats_all, threshold=pcc_th)
    n_pcc = len(feats_pcc)
    if n_pcc == 0:
        raise RuntimeError(f"{margin}: PCC left 0 features on Train.")

    # --- RF-IFS on full Train (用你现成的方法；内部CV只用Train) ---
    feats_rf = pipe._rf_ifs_select(feats_pcc, name=f"ValTest_{margin}")
    n_rf = len(feats_rf)
    if n_rf == 0:
        raise RuntimeError(f"{margin}: RF-IFS left 0 features on Train.")

    # --- LASSO(CV选C) on full Train, predict Val ---
    X_tr = df_tr[feats_rf].apply(pd.to_numeric, errors="coerce").values
    y_tr = df_tr["Target"].values.astype(int)
    X_va = df_va[feats_rf].apply(pd.to_numeric, errors="coerce").values
    y_va = df_va["Target"].values.astype(int)

    imp = SimpleImputer(strategy="mean")
    X_tr = imp.fit_transform(X_tr)
    X_va = imp.transform(X_va)

    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_va = sc.transform(X_va)

    hard_Cs = np.logspace(-4, -0.3, 20)
    mdl = LogisticRegressionCV(
        Cs=hard_Cs,
        cv=5,
        penalty="l1",
        solver="liblinear",
        scoring="roc_auc",
        max_iter=10000,
        random_state=seed,
        n_jobs=-1
    )
    mdl.fit(X_tr, y_tr)
    p_va = mdl.predict_proba(X_va)[:, 1]
    auc_va = float(roc_auc_score(y_va, p_va))

    # --- Extract selected (non-zero) features ---
    coefs = mdl.coef_.ravel()
    idx = np.where(coefs != 0)[0]
    selected_feats = [feats_rf[i] for i in idx]
    n_lasso = len(selected_feats)

    # 为了便于导出：系数表
    coef_table = None
    if n_lasso > 0:
        coef_table = pd.DataFrame({
            "feature": selected_feats,
            "coef": [coefs[i] for i in idx]
        }).sort_values("coef", key=lambda s: np.abs(s), ascending=False)

    return {
        "margin": margin,
        "y_val": y_va,
        "p_val": p_va,
        "auc_val": auc_va,
        "n_all": len(feats_all),
        "n_pcc": n_pcc,
        "n_rf": n_rf,
        "n_lasso": n_lasso,
        "selected_feats": selected_feats,
        "coef_table": coef_table
    }

# =========================
# 主流程：三组 margin 训练并预测 Val
# =========================
SEED = 42
PCC_TH = 0.8
N_BOOT = 5000

results = {}
for m in ["2mm", "3mm", "4mm"]:
    print(f"\n=== Training on Train, testing on Val: {m} ===")
    results[m] = train_pipeline_predict_val(pipe, margin=m, pcc_th=PCC_TH, seed=SEED)

# =========================
# 汇总：Val AUC + bootstrap CI/SD
# =========================
rows = []
y_val = results["4mm"]["y_val"]  # 三者相同
for m in ["2mm", "3mm", "4mm"]:
    auc_point, auc_boot_mean, auc_boot_sd, ci_low, ci_high = bootstrap_auc_ci(
        y_val, results[m]["p_val"], n_boot=N_BOOT, seed=SEED
    )
    rows.append({
        "Margin": m,
        "Val AUC (point)": auc_point,
        "Val AUC (boot mean)": auc_boot_mean,
        "Val AUC (boot SD)": auc_boot_sd,
        "Val AUC 95% CI low": ci_low,
        "Val AUC 95% CI high": ci_high,
        "n_all_feats": results[m]["n_all"],
        "n_after_PCC": results[m]["n_pcc"],
        "n_after_RFIFS": results[m]["n_rf"],
        "n_after_LASSO": results[m]["n_lasso"],
    })

summary = pd.DataFrame(rows)
summary["Val AUC ± SD (bootstrap)"] = summary.apply(
    lambda r: f"{r['Val AUC (point)']:.3f} ± {r['Val AUC (boot SD)']:.3f}", axis=1
)

print("\n=== Val AUC summary ===")
print(summary[[
    "Margin",
    "Val AUC (point)",
    "Val AUC ± SD (bootstrap)",
    "Val AUC 95% CI low",
    "Val AUC 95% CI high",
    "n_after_PCC",
    "n_after_RFIFS",
    "n_after_LASSO"
]].to_string(index=False))

# =========================
# 差异检验：在 Val 上比较 4mm vs 2mm/3mm
# =========================
tests_lines = []
p4 = results["4mm"]["p_val"]

for comp in ["2mm", "3mm"]:
    pc = results[comp]["p_val"]

    auc4, aucc, p_delong = delong_pvalue(y_val, p4, pc)
    diff_mean, ci_low, ci_high, p_boot = bootstrap_auc_diff(
        y_val, p4, pc, n_boot=N_BOOT, seed=SEED
    )

    line = (
        f"=== 4mm vs {comp} (Val) ===\n"
        f"AUC(4mm)={auc4:.4f} | AUC({comp})={aucc:.4f} | ΔAUC={auc4-aucc:+.4f}\n"
        f"DeLong p-value = {p_delong:.4g}\n"
        f"Bootstrap ΔAUC mean = {diff_mean:+.4f} | 95% CI [{ci_low:+.4f}, {ci_high:+.4f}] | p = {p_boot:.4g}\n"
    )
    tests_lines.append(line)
    print("\n" + line)

# =========================
# 输出到results文件夹
# =========================
out_xlsx = os.path.join(BASE_DIR, "Val_Margin_AUC_Summary.xlsx")
out_txt  = os.path.join(BASE_DIR, "Val_Margin_AUC_Tests.txt")
out_csv  = os.path.join(BASE_DIR, "Val_Margin_Preds.csv")
out_coef_dir = os.path.join(BASE_DIR, "Val_Margin_Coefs")
os.makedirs(out_coef_dir, exist_ok=True)

# 1) summary excel
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
    summary.to_excel(w, index=False, sheet_name="Val_AUC_Summary")

print(f"\nSaved: {out_xlsx}")

# 2) tests txt
with open(out_txt, "w", encoding="utf-8") as f:
    f.write("=== Val AUC summary ===\n")
    f.write(summary.to_string(index=False))
    f.write("\n\n")
    for line in tests_lines:
        f.write(line + "\n")

print(f"Saved: {out_txt}")

# 3) preds csv
if EXPORT_PATIENT_LEVEL_PREDICTIONS:
    pred_df = pd.DataFrame({
        "ID": pipe.val_data.index.astype(str),
        "y_val": y_val,
        "p_2mm": results["2mm"]["p_val"],
        "p_3mm": results["3mm"]["p_val"],
        "p_4mm": results["4mm"]["p_val"],
    })
    pred_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"Saved restricted patient-level predictions: {out_csv}")
else:
    print("Skipped patient-level margin predictions (EXPORT_PATIENT_LEVEL_PREDICTIONS=False).")

# 4) 每个 margin 的 LASSO 非零系数表
for m in ["2mm", "3mm", "4mm"]:
    coef_table = results[m]["coef_table"]
    if coef_table is not None and len(coef_table) > 0:
        coef_path = os.path.join(out_coef_dir, f"{m}_LASSO_nonzero_coefs.csv")
        coef_table.to_csv(coef_path, index=False, encoding="utf-8-sig")
        print(f"Saved: {coef_path}")
    else:
        print(f"{m}: no non-zero LASSO features to save.")


## 4. Final score coefficients (Supplementary Table S2A)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV

# ====== results输出路径 ======
BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)
OUT_XLSX = os.path.join(BASE_DIR, "Signature_Features_Coefficients.xlsx")

SEED = 42
PCC_TH = 0.8

def fit_signature_and_extract(df_tr, df_va, feat_cols, name, seed=42):
    """
    复现你 _fit_lasso_scores 中的“全Train拟合最终模型并提取系数”的部分：
    Mean imputation + StandardScaler + LogisticRegressionCV(L1)
    返回：coef表（含标准化空间 coef_std 与还原尺度 coef_raw）+ intercept 等信息
    """
    X_tr = df_tr[feat_cols].apply(pd.to_numeric, errors="coerce").values
    y_tr = df_tr["Target"].values.astype(int)
    X_va = df_va[feat_cols].apply(pd.to_numeric, errors="coerce").values

    # 与你代码一致：mean impute
    imp = SimpleImputer(strategy="mean")
    X_tr_imp = imp.fit_transform(X_tr)
    X_va_imp = imp.transform(X_va)

    # 与你代码一致：standardize（用 Train 拟合）
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_imp)
    X_va_sc = sc.transform(X_va_imp)

    hard_Cs = np.logspace(-4, -0.3, 20)

    mdl = LogisticRegressionCV(
        Cs=hard_Cs,
        cv=5,
        penalty="l1",
        solver="liblinear",
        scoring="roc_auc",
        max_iter=10000,
        random_state=seed,
        n_jobs=-1
    )
    mdl.fit(X_tr_sc, y_tr)

    # 提取系数（标准化空间）
    coef_std = mdl.coef_.ravel()
    intercept_std = float(mdl.intercept_.ravel()[0])

    # 非零特征
    idx = np.where(coef_std != 0)[0]
    selected_feats = [feat_cols[i] for i in idx]
    selected_coef_std = coef_std[idx]

    # ====== 还原到“标准化前（impute 后原始尺度）”的系数 ======
    # 若 z_j = (x_j - mean_j)/scale_j
    # η = b0 + Σ b_j z_j = (b0 - Σ b_j*mean_j/scale_j) + Σ (b_j/scale_j) x_j
    scale = sc.scale_.ravel()
    mean = sc.mean_.ravel()

    coef_raw = selected_coef_std / scale[idx]
    intercept_raw = intercept_std - float(np.sum(selected_coef_std * (mean[idx] / scale[idx])))

    # 生成表
    df_coef = pd.DataFrame({
        "feature": selected_feats,
        "coef_std (on z)": selected_coef_std,
        "coef_raw (on x_imp)": coef_raw,
        "abs(coef_std)": np.abs(selected_coef_std),
    }).sort_values("abs(coef_std)", ascending=False).reset_index(drop=True)

    meta = {
        "name": name,
        "n_input_feats": len(feat_cols),
        "n_nonzero": len(selected_feats),
        "intercept_std": intercept_std,
        "intercept_raw": intercept_raw,
    }
    return df_coef, meta

def build_feature_list_for_modality(pipe, modality):
    """
    按你 build_signatures 的逻辑构造特征列表（基于 best_margin）
    modality: 'Rad' / 'DL' / 'Hab'
    """
    if getattr(pipe, "best_margin", None) is None:
        raise RuntimeError("pipe.best_margin is None. 请先运行 pipe.run_margin_tournament() 并确保 best_margin 已确定。")

    best_p = f"p{pipe.best_margin[0]}"
    cols = pipe.train_data.columns

    if modality == "Rad":
        feats = [c for c in cols if c.startswith("t_Rad_") or c.startswith(f"{best_p}_Rad_")]
    elif modality == "DL":
        feats = [c for c in cols if c.startswith("t_DL_") or c.startswith(f"{best_p}_DL_")]
    elif modality == "Hab":
        feats = [c for c in cols if c.startswith("Hab_t_") or c.startswith(f"Hab_{best_p}_")]
    else:
        raise ValueError("modality must be one of: 'Rad','DL','Hab'")
    return feats

# ====== 主导出逻辑 ======
if "pipe" not in globals():
    raise RuntimeError("当前 kernel 没有 pipe。请先运行你的主 pipeline 生成 pipe（至少 load_data + run_margin_tournament）。")

df_tr = pipe.train_data
df_va = pipe.val_data

export_tables = {}
export_meta = []

for modality in ["Rad", "DL", "Hab"]:
    print(f"\n=== Exporting {modality}-score coefficients ===")

    # 1) 根据 margin 取候选
    feats0 = build_feature_list_for_modality(pipe, modality)

    # 2) PCC -> RF-IFS（与 build_signatures 一致；都只基于 Train）
    feats_pcc = pipe._pcc_reduction(feats0, threshold=PCC_TH)
    feats_rf = pipe._rf_ifs_select(feats_pcc, modality)

    # 3) 复现最终 LASSO 模型并提取系数（全Train拟合）
    df_coef, meta = fit_signature_and_extract(df_tr, df_va, feats_rf, f"{modality}-score", seed=SEED)

    export_tables[modality] = df_coef
    export_meta.append(meta)

    print(f"Input feats: {meta['n_input_feats']} | Non-zero: {meta['n_nonzero']}")
    print(df_coef.head(10).to_string(index=False))

# ====== 输出到 Excel（results） ======
meta_df = pd.DataFrame(export_meta)
formula_df = pd.DataFrame({
    "Item": [
        "Imputation",
        "Standardization",
        "Linear predictor (logit)",
        "Probability score",
        "Raw-scale coefficients conversion (after imputation)"
    ],
    "Description/LaTeX": [
        r"$\tilde{x}_{ij} = x_{ij}$ if observed, else $\tilde{x}_{ij} = \mu_j$ (train mean)",
        r"$z_{ij} = (\tilde{x}_{ij}-\bar{x}_j)/s_j$ (mean/std from train after imputation)",
        r"$\eta_i = \beta_0 + \sum_{j=1}^{p}\beta_j z_{ij}$",
        r"$\text{score}_i = \sigma(\eta_i)=\dfrac{1}{1+e^{-\eta_i}}$",
        r"$\alpha_j=\beta_j/s_j,\ \alpha_0=\beta_0-\sum_j \beta_j\bar{x}_j/s_j,\ \eta_i=\alpha_0+\sum_j\alpha_j\tilde{x}_{ij}$"
    ]
})

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    meta_df.to_excel(w, index=False, sheet_name="Meta")
    formula_df.to_excel(w, index=False, sheet_name="Formula")
    export_tables["Rad"].to_excel(w, index=False, sheet_name="Rad_score")
    export_tables["DL"].to_excel(w, index=False, sheet_name="DL_score")
    export_tables["Hab"].to_excel(w, index=False, sheet_name="Hab_score")

print(f"\nSaved Excel to: {OUT_XLSX}")


## 5. Clinical-variable screening

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ========= 输出路径（默认results；若你主脚本已定义 BASE_DIR，会自动沿用）=========
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

OUT_XLSX = os.path.join(BASE_DIR, "Table1_Clinical_Baseline_Univariate_FDR.xlsx")

# ========= BH-FDR q values（对“变量层面”的 P 值做校正）=========
def bh_qvalues(pval_dict):
    s = pd.Series(pval_dict, dtype=float)
    valid = s.dropna()
    if valid.empty:
        return {k: np.nan for k in s.index}

    m = len(valid)
    p_sorted = valid.sort_values()
    ranks = np.arange(1, m + 1, dtype=float)

    q_sorted = (p_sorted.values * m / ranks)
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q_sorted = np.clip(q_sorted, 0, 1)

    q_series = pd.Series(q_sorted, index=p_sorted.index)
    q_all = s.index.to_series().map(q_series).astype(float)
    return {k: (float(q_all.loc[k]) if pd.notna(q_all.loc[k]) else np.nan) for k in s.index}

# ========= Table 1 生成（按列顺序、分类变量展开水平行）=========
def build_table1_clinical(df, target_col="Target", q=0.05, cat_unique_cut=5):
    # 与你 pipeline 一致：精确排除 + 排除影像/生境前缀
    exclude_exact = set(['Target', 'Group', 'Rad_score', 'DL_score', 'Hab_score', 'LNM', 'N0'])

    candidates = []
    for c in df.columns:
        if c in exclude_exact:
            continue
        if c.startswith(('t_', 'p2_', 'p3_', 'p4_', 'Hab_')):
            continue
        candidates.append(c)

    y = df[target_col].astype(int)
    g0 = df.loc[y == 0]
    g1 = df.loc[y == 1]
    n0, n1 = len(g0), len(g1)

    # 变量层面 P 值（用于 FDR），分类变量按“变量”算一次 P
    pvals = {}

    # 最终 Table 1 行
    rows = []

    def _fmt_mean_sd(x):
        return f"{np.nanmean(x):.3g} ± {np.nanstd(x, ddof=1):.3g}"

    def _fmt_median_iqr(x):
        x = x[~np.isnan(x)]
        if len(x) == 0:
            return ""
        q1, med, q3 = np.percentile(x, [25, 50, 75])
        return f"{med:.3g} [{q1:.3g}, {q3:.3g}]"

    def _is_normal_shapiro(x):
        x = x[~np.isnan(x)]
        # Shapiro 对样本太少没意义；这里做一个基本保护
        if len(x) < 3:
            return False, np.nan
        try:
            stat, p = stats.shapiro(x)
            return (p >= 0.05), float(p)
        except Exception:
            return False, np.nan

    def _cat_levels_in_order(x_nonmissing):
        # 保持“出现顺序”，更像临床表
        seen = []
        for v in x_nonmissing:
            if v not in seen:
                seen.append(v)
        return seen

    for col in candidates:
        x_raw = df[col]
        miss0 = int(g0[col].isna().sum())
        miss1 = int(g1[col].isna().sum())
        miss_all = int(x_raw.isna().sum())
        nunq = int(x_raw.nunique(dropna=True))

        # 判定离散/连续（与你筛选逻辑一致：<=5 当类别）
        is_cat = (nunq <= cat_unique_cut)

        if is_cat:
            # —— 分类变量：水平展开行 ——
            # 用“非缺失”来统计 n(%)，更符合期刊 Table 1
            x0 = g0[col].dropna()
            x1 = g1[col].dropna()

            # 计算 P 值：优先卡方；2x2 且期望频数过小可用 Fisher
            # 构建列联表（剔除缺失）
            x_all = df[col].dropna()
            yy_all = y.loc[x_all.index]
            ct = pd.crosstab(x_all, yy_all)

            test_name = "Chi-square"
            p = np.nan
            try:
                if ct.shape[0] > 1 and ct.shape[1] > 1:
                    chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
                    # 若 2x2 且期望频数<5，使用 Fisher（更期刊化）
                    if ct.shape == (2, 2) and np.any(expected < 5):
                        odds, p_f = stats.fisher_exact(ct.values)
                        p = float(p_f)
                        test_name = "Fisher exact"
                    else:
                        p = float(p_chi)
                else:
                    p = 1.0
            except Exception:
                p = np.nan

            pvals[col] = p

            # 水平顺序：按出现顺序
            levels = _cat_levels_in_order(x_all.values.tolist())

            # 分母：各组非缺失样本量
            denom0 = len(x0)
            denom1 = len(x1)

            # 为了 Table 1 美观：变量名只在第一行显示
            first = True
            for lv in levels:
                c0 = int((x0 == lv).sum())
                c1 = int((x1 == lv).sum())
                s0 = f"{c0} ({(c0/denom0*100):.1f}%)" if denom0 > 0 else ""
                s1 = f"{c1} ({(c1/denom1*100):.1f}%)" if denom1 > 0 else ""

                rows.append({
                    "Variable": col if first else "",
                    "Level": str(lv),
                    f"Group0 (n={n0})": s0,
                    f"Group1 (n={n1})": s1,
                    "Missing0": miss0 if first else "",
                    "Missing1": miss1 if first else "",
                    "Test": test_name if first else "",
                    "P_value": p if first else "",
                    "FDR_q": "",  # 稍后填
                    "Pass_FDR(q<0.05)": "",
                    "Type": "Categorical" if first else "",
                })
                first = False

        else:
            # —— 连续变量：Table 1 常见展示（正态：均值±SD；非正态：中位数[IQR]）——
            x0 = pd.to_numeric(g0[col], errors="coerce").values.astype(float)
            x1 = pd.to_numeric(g1[col], errors="coerce").values.astype(float)
            x_all = pd.to_numeric(df[col], errors="coerce").values.astype(float)

            # 描述统计展示格式：用整体 Shapiro（更简洁）
            is_norm, p_norm = _is_normal_shapiro(x_all)
            if is_norm:
                s0 = _fmt_mean_sd(x0)
                s1 = _fmt_mean_sd(x1)
                disp = "Mean ± SD"
            else:
                s0 = _fmt_median_iqr(x0)
                s1 = _fmt_median_iqr(x1)
                disp = "Median [IQR]"

            # P 值：按你方法学（连续用 Mann–Whitney U）
            test_name = "Mann–Whitney U"
            p = np.nan
            try:
                xx0 = x0[~np.isnan(x0)]
                xx1 = x1[~np.isnan(x1)]
                if len(xx0) > 0 and len(xx1) > 0:
                    _, p = stats.mannwhitneyu(xx0, xx1)
                    p = float(p)
                else:
                    p = np.nan
            except Exception:
                p = np.nan

            pvals[col] = p

            rows.append({
                "Variable": col,
                "Level": disp,
                f"Group0 (n={n0})": s0,
                f"Group1 (n={n1})": s1,
                "Missing0": miss0,
                "Missing1": miss1,
                "Test": test_name,
                "P_value": p,
                "FDR_q": "",  # 稍后填
                "Pass_FDR(q<0.05)": "",
                "Type": "Continuous",
            })

    # —— 计算 FDR q，并回填（按变量层面）——
    qvals = bh_qvalues(pvals)

    for r in rows:
        var = r["Variable"] if r["Variable"] != "" else None
        # 只有变量首行才写 q / pass（首行 Variable 非空）
        if var:
            qv = qvals.get(var, np.nan)
            r["FDR_q"] = qv
            r["Pass_FDR(q<0.05)"] = (qv is not None and not np.isnan(qv) and qv < q)
        else:
            r["FDR_q"] = ""
            r["Pass_FDR(q<0.05)"] = ""

    table1 = pd.DataFrame(rows)

    # 选入变量：q<0.05 的“变量名”
    passed = []
    for v in candidates:
        qv = qvals.get(v, np.nan)
        if qv is not None and not np.isnan(qv) and qv < q:
            passed.append(v)

    # 与你 pipeline 一致的 fallback
    selected_for_fusion = passed if passed else (["Age"] if "Age" in candidates else [])

    # 再输出一个“变量层面”简表（便于你在 Methods/Results 引用）
    var_level = pd.DataFrame({
        "Variable": candidates,
        "P_value": [pvals.get(v, np.nan) for v in candidates],
        "FDR_q": [qvals.get(v, np.nan) for v in candidates],
        "Pass_FDR(q<0.05)": [(qvals.get(v, np.nan) < q) if pd.notna(qvals.get(v, np.nan)) else False for v in candidates],
    })

    return table1, var_level, selected_for_fusion

# ========= 运行并导出 =========
if "pipe" not in globals():
    raise RuntimeError("当前 kernel 没有 pipe。请先运行你的主 pipeline，确保 pipe.load_data() 已执行。")

table1, var_level, selected_vars = build_table1_clinical(pipe.train_data, target_col="Target", q=0.05)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    table1.to_excel(w, index=False, sheet_name="Table1")
    var_level.to_excel(w, index=False, sheet_name="Variable_Level")
    pd.DataFrame({"Selected_for_fusion": selected_vars}).to_excel(w, index=False, sheet_name="Selected")
    pd.DataFrame({
        "Notes": [
            "Categorical variables: shown as n (%), computed within non-missing in each group.",
            "Continuous variables: display as Mean ± SD if Shapiro–Wilk p≥0.05; otherwise Median [IQR].",
            "P-values: categorical by Chi-square (or Fisher exact for 2x2 with small expected counts); continuous by Mann–Whitney U.",
            "FDR q-values: Benjamini–Hochberg correction across variables (variable-level p-values).",
        ]
    }).to_excel(w, index=False, sheet_name="Notes")

# ========= 简单格式美化（列宽、冻结首行、表头加粗、自动换行）=========
wb = load_workbook(OUT_XLSX)
ws = wb["Table1"]

# 冻结首行
ws.freeze_panes = "A2"

# 表头加粗 + 居中
header_font = Font(bold=True)
for cell in ws[1]:
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# 自动列宽（按内容长度粗略估计）
for col_idx, col_cells in enumerate(ws.columns, start=1):
    max_len = 0
    for c in col_cells:
        try:
            v = "" if c.value is None else str(c.value)
            max_len = max(max_len, len(v))
        except Exception:
            pass
    width = min(max(10, max_len + 2), 60)
    ws.column_dimensions[get_column_letter(col_idx)].width = width

# 内容对齐：文本左、数值居中（简单处理）
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for c in row:
        if c.column_letter in ["A", "B", "J"]:  # Variable/Level/Type
            c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
        else:
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

wb.save(OUT_XLSX)

# 写回 pipe，方便后续 fusion 用
pipe.clinical_table1 = table1
pipe.clinical_univariate_varlevel = var_level
pipe.selected_clin_vars = selected_vars

print("Saved:", OUT_XLSX)
print("Selected clinical vars for fusion:", selected_vars)
display(table1.head(15))


## 6. Progressive-model evaluation (Figure 4 and Table 2)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ===== 路径 =====
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

# ===== 基础检查 =====
assert "pipe" in globals(), "未找到 pipe。请先运行你的主 pipeline（至少 load_data + build_signatures + step_select_clinical_features）。"
for col in ["Target", "Rad_score", "DL_score", "Hab_score"]:
    assert col in pipe.train_data.columns, f"pipe.train_data 缺少列：{col}。请先运行 build_signatures()。"

assert hasattr(pipe, "selected_clin_vars"), "未找到 pipe.selected_clin_vars。请先运行 step_select_clinical_features()。"
clin = pipe.selected_clin_vars

# ===== 模型定义（与你 Step 4 一致）=====
models = {
    "Model 1 (Clin Only)": clin,
    "Model 2 (+Rad)": clin + ["Rad_score"],
    "Model 3 (+DL)": clin + ["Rad_score", "DL_score"],
    "Model 4 (+Hab)": clin + ["Rad_score", "DL_score", "Hab_score"],
}

SEED = 42
N_SPLITS = 5

def _fit_predict_lr(train_df, val_df, feats, seed=42, n_splits=5):
    """
    输出：
      - train_oof: 训练集OOF预测（更合理的“训练集ROC”）
      - train_fit: 用全训练集拟合后在训练集上的预测（偏乐观，可选）
      - val_pred:  用全训练集拟合后在验证集的预测
    预处理与 Step4 一致：mean impute + standardize + LR
    """
    X_tr = train_df[feats].apply(pd.to_numeric, errors="coerce").values
    y_tr = train_df["Target"].values.astype(int)
    X_va = val_df[feats].apply(pd.to_numeric, errors="coerce").values
    y_va = val_df["Target"].values.astype(int)

    # ---- OOF ----
    oof = np.zeros(len(train_df), dtype=float)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr_idx, te_idx in cv.split(X_tr, y_tr):
        X_tr_fold, X_te_fold = X_tr[tr_idx], X_tr[te_idx]
        y_tr_fold = y_tr[tr_idx]

        imp = SimpleImputer(strategy="mean")
        X_tr_fold = imp.fit_transform(X_tr_fold)
        X_te_fold = imp.transform(X_te_fold)

        sc = StandardScaler()
        X_tr_fold = sc.fit_transform(X_tr_fold)
        X_te_fold = sc.transform(X_te_fold)

        lr = LogisticRegression(random_state=seed, max_iter=5000)
        lr.fit(X_tr_fold, y_tr_fold)
        oof[te_idx] = lr.predict_proba(X_te_fold)[:, 1]

    # ---- Fit on full Train -> predict Train & Val ----
    imp = SimpleImputer(strategy="mean")
    X_tr_imp = imp.fit_transform(X_tr)
    X_va_imp = imp.transform(X_va)

    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_imp)
    X_va_sc = sc.transform(X_va_imp)

    lr = LogisticRegression(random_state=seed, max_iter=5000)
    lr.fit(X_tr_sc, y_tr)

    train_fit = lr.predict_proba(X_tr_sc)[:, 1]
    val_pred = lr.predict_proba(X_va_sc)[:, 1]

    return y_tr, y_va, oof, train_fit, val_pred

# ===== 计算所有模型预测并汇总AUC =====
pred_store = {}
rows = []

for name, feats in models.items():
    y_tr, y_va, p_tr_oof, p_tr_fit, p_va = _fit_predict_lr(
        pipe.train_data, pipe.val_data, feats, seed=SEED, n_splits=N_SPLITS
    )

    auc_tr_oof = roc_auc_score(y_tr, p_tr_oof)
    auc_tr_fit = roc_auc_score(y_tr, p_tr_fit)
    auc_va = roc_auc_score(y_va, p_va)

    rows.append({
        "Model": name,
        "AUC Train (OOF)": auc_tr_oof,
        "AUC Train (fit)": auc_tr_fit,
        "AUC Val": auc_va,
        "Gap (OOF-Val)": auc_tr_oof - auc_va,
        "Features": ", ".join(feats),
    })

    pred_store[name] = {
        "y_train": y_tr, "y_val": y_va,
        "p_train_oof": p_tr_oof,
        "p_train_fit": p_tr_fit,
        "p_val": p_va
    }

auc_table = pd.DataFrame(rows)
display(auc_table)

# ===== 导出：AUC表 + 预测概率 =====
out_auc_xlsx = os.path.join(BASE_DIR, "Fusion_AUC_Summary.xlsx")
out_pred_csv  = os.path.join(BASE_DIR, "Fusion_Predictions.csv")

auc_table.to_excel(out_auc_xlsx, index=False)

# Patient-level prediction exports are optional and disabled by default.
if EXPORT_PATIENT_LEVEL_PREDICTIONS:
    train_df_out = pd.DataFrame({
        "ID": pipe.train_data.index.astype(str),
        "y": pred_store[list(models.keys())[0]]["y_train"],
    })
    val_df_out = pd.DataFrame({
        "ID": pipe.val_data.index.astype(str),
        "y": pred_store[list(models.keys())[0]]["y_val"],
    })
    for name in models.keys():
        key = name.replace(" ", "_").replace("(", "").replace(")", "").replace("+", "plus")
        train_df_out[f"p_{key}_OOF"] = pred_store[name]["p_train_oof"]
        train_df_out[f"p_{key}_FIT"] = pred_store[name]["p_train_fit"]
        val_df_out[f"p_{key}"] = pred_store[name]["p_val"]
    pred_all = pd.concat([
        train_df_out.assign(Set="Train"),
        val_df_out.assign(Set="Val"),
    ], axis=0, ignore_index=True)
    pred_all.to_csv(out_pred_csv, index=False, encoding="utf-8-sig")
    print("Saved restricted patient-level predictions:", out_pred_csv)
else:
    print("Skipped patient-level fusion predictions (EXPORT_PATIENT_LEVEL_PREDICTIONS=False).")


In [ ]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# ========= 论文级输出建议参数 =========
mpl.rcParams.update({
    "pdf.fonttype": 42,   # TrueType 字体嵌入，避免PDF里字体变成Type3
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

# 输出文件（PNG + PDF）
out_png_val_all   = os.path.join(BASE_DIR, "ROC_Val_AllModels.png")
out_pdf_val_all   = os.path.join(BASE_DIR, "ROC_Val_AllModels.pdf")

out_png_train_all = os.path.join(BASE_DIR, "ROC_TrainOOF_AllModels.png")
out_pdf_train_all = os.path.join(BASE_DIR, "ROC_TrainOOF_AllModels.pdf")

# 你可以按需要调整：PNG 分辨率（论文插图一般 600 dpi 起）
DPI_PNG = 600

# ===== 图1：验证集 ROC（四模型）=====
fig, ax = plt.subplots(figsize=(7.2, 6.2))
for name in models.keys():
    y = pred_store[name]["y_val"]
    p = pred_store[name]["p_val"]
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    ax.plot(fpr, tpr, lw=2.2, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.2)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Validation ROC Curves (Fusion Models)")
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()

fig.savefig(out_png_val_all, dpi=DPI_PNG)
fig.savefig(out_pdf_val_all)  # PDF为矢量图，不需要dpi
plt.show()
plt.close(fig)
print("Saved:", out_png_val_all)
print("Saved:", out_pdf_val_all)

# ===== 图2：训练集（OOF）ROC（四模型）=====
fig, ax = plt.subplots(figsize=(7.2, 6.2))
for name in models.keys():
    y = pred_store[name]["y_train"]
    p = pred_store[name]["p_train_oof"]  # 训练集用 OOF，避免乐观偏倚
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    ax.plot(fpr, tpr, lw=2.2, label=f"{name} (OOF AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.2)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Training ROC Curves (OOF, Fusion Models)")
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()

fig.savefig(out_png_train_all, dpi=DPI_PNG)
fig.savefig(out_pdf_train_all)
plt.show()
plt.close(fig)
print("Saved:", out_png_train_all)
print("Saved:", out_pdf_train_all)


## 7. Calibration and decision curves (Figure 5)

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# ========= 论文级输出建议参数 =========
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

# ========= 核心修正：名称必须与 pred_store 中的 Key 完全一致 =========
# 你的 pred_store 中使用的是：
# "Model 1 (Clin Only)", "Model 2 (+Rad)", "Model 3 (+DL)", "Model 4 (+Hab)"
model_configs = [
    {"key": "Model 1 (Clin Only)", "label": "Model 1", "color": "blue"},
    {"key": "Model 2 (+Rad)",      "label": "Model 2", "color": "orange"},
    {"key": "Model 3 (+DL)",       "label": "Model 3", "color": "green"},
    {"key": "Model 4 (+Hab)",      "label": "Model 4", "color": "red"} 
]

assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"

# 校准曲线参数
n_bins = 7
strategy = "quantile"

# 输出文件
out_png_cal = os.path.join(BASE_DIR, "Calibration_AllModels_Val.png")
out_pdf_cal = os.path.join(BASE_DIR, "Calibration_AllModels_Val.pdf")
DPI_PNG = 600

fig, ax = plt.subplots(figsize=(7.2, 6.2))

# 1. 先画 Perfect 线 (黑色虚线)
ax.plot([0, 1], [0, 1], "--", color="black", linewidth=1.5, label="Perfect")

# 2. 循环画出 Model 1 ~ 4
for config in model_configs:
    m_key = config["key"]      # 用于从字典取数据，必须全名
    m_label = config["label"]  # 用于图例显示，可以用短名
    m_color = config["color"]
    
    if m_key in pred_store:
        y_val = pred_store[m_key]["y_val"]
        p_val = pred_store[m_key]["p_val"]
        
        # 计算校准曲线
        frac_pos, mean_pred = calibration_curve(y_val, p_val, n_bins=n_bins, strategy=strategy)
        
        # 绘图
        ax.plot(mean_pred, frac_pos, "o-", linewidth=2, color=m_color, label=m_key) # 图例如果想显示短名，把 label=m_key 改为 label=m_label
    else:
        print(f"Warning: {m_key} not found in pred_store, skipping...")

ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Actual Fraction")
ax.set_title("Calibration Curve (Validation)")
ax.legend(loc="upper left", frameon=False)
fig.tight_layout()

fig.savefig(out_png_cal, dpi=DPI_PNG)
fig.savefig(out_pdf_cal)
plt.show()
plt.close(fig)

print("Saved:", out_png_cal)
print("Saved:", out_pdf_cal)

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ========= 论文级输出建议参数（不改变颜色） =========
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

# ========= 配置部分：定义模型 Key 和颜色 =========
# 对应 Model 1 ~ 4，颜色：Blue, Orange, Green, Red
model_configs = [
    {"key": "Model 1 (Clin Only)", "color": "blue",   "label": "Model 1 (Clin Only)"},
    {"key": "Model 2 (+Rad)",      "color": "orange", "label": "Model 2 (+Rad)"},
    {"key": "Model 3 (+DL)",       "color": "green",  "label": "Model 3 (+DL)"},
    {"key": "Model 4 (+Hab)",      "color": "red",    "label": "Model 4 (+Hab)"}
]

assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"

# ===== DCA 计算函数 =====
def decision_curve(y_true, p_pred, thresholds):
    """
    Net benefit = TP/N - FP/N * (pt/(1-pt))
    """
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    N = len(y_true)

    nb = np.zeros_like(thresholds, dtype=float)
    for i, pt in enumerate(thresholds):
        pred_pos = (p_pred >= pt)
        TP = np.sum(pred_pos & (y_true == 1))
        FP = np.sum(pred_pos & (y_true == 0))
        nb[i] = TP / N - FP / N * (pt / (1 - pt))
    return nb

# 输出目录
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

# 输出文件（包含所有模型）
out_png = os.path.join(BASE_DIR, "DCA_AllModels_Val.png")
out_pdf = os.path.join(BASE_DIR, "DCA_AllModels_Val.pdf")
DPI_PNG = 600

# ===== 准备绘图 =====
fig, ax = plt.subplots(figsize=(8.6, 7.2))
thresholds = np.linspace(0.01, 0.99, 99)

# 1. 获取基础数据 (y_val) 以计算 Treat All / Treat None
# 随便取一个存在的模型来获取 y_val (所有模型的 y_val 应该是一样的)
first_key = model_configs[3]["key"] # 尝试取 Model 4
if first_key not in pred_store:
    # 如果 Model 4 不在，尝试取第一个
    first_key = model_configs[0]["key"]

if first_key in pred_store:
    y_val_common = np.asarray(pred_store[first_key]["y_val"]).astype(int)
    
    # --- Treat All / Treat None ---
    prev = y_val_common.mean()
    nb_none = np.zeros_like(thresholds)
    nb_all = prev - (1 - prev) * (thresholds / (1 - thresholds))

    # 绘制参考线 (Treat All / None) - 放在底层
    ax.plot(thresholds, nb_all, color="grey", linestyle="--", linewidth=2, label="Treat All")
    ax.plot(thresholds, nb_none, color="black", linewidth=2, label="Treat None")
else:
    print("Error: 无法找到任何模型数据来计算 y_val")

# 2. 循环绘制 Model 1 ~ 4
for config in model_configs:
    m_key = config["key"]
    m_color = config["color"]
    m_label = config["label"] # 图例显示的名称

    if m_key in pred_store:
        y_val = pred_store[m_key]["y_val"]
        p_val = pred_store[m_key]["p_val"]
        
        # 计算该模型的 Net Benefit
        nb_model = decision_curve(y_val, p_val, thresholds)
        
        # 绘图
        ax.plot(thresholds, nb_model, color=m_color, linewidth=2, label=m_label)
    else:
        print(f"Warning: {m_key} not found in pred_store, skipping...")

# ===== 图像修饰 =====
ax.set_title("Decision Curve Analysis")
ax.set_xlabel("Threshold Probability")
ax.set_ylabel("Net Benefit")

# 轴范围对齐 (根据你原代码保留)
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 0.5)

ax.legend(loc="upper right", frameon=False)
fig.tight_layout()

# 保存
fig.savefig(out_png, dpi=DPI_PNG)
fig.savefig(out_pdf) 
plt.show()
plt.close(fig)

print("Saved:", out_png)
print("Saved:", out_pdf)

## 8. Reclassification and pairwise comparisons (Table 3 and Table S3)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
from scipy.stats import norm

# ================= 配置 =================
BOOTSTRAP_ROUNDS = 1000  # 重采样次数
SEED = 42
OUTPUT_FILE = os.path.join(BASE_DIR, "NRI_IDI_Statistics.xlsx")  # 输出文件名
# =======================================

def calculate_nri_idi_point_estimate(y_true, p_ref, p_new):
    """计算单次 NRI 和 IDI 的点估计值 (用于 Bootstrap 内部调用)"""
    y_true = np.array(y_true)
    p_ref = np.array(p_ref)
    p_new = np.array(p_new)
    
    # --- IDI ---
    diff = p_new - p_ref
    idi_events = np.mean(diff[y_true == 1])
    idi_nonevents = np.mean(diff[y_true == 0])
    idi = idi_events - idi_nonevents
    
    # --- NRI (Continuous) ---
    # Events
    up_events = np.sum(p_new[y_true == 1] > p_ref[y_true == 1])
    down_events = np.sum(p_new[y_true == 1] < p_ref[y_true == 1])
    n_events = np.sum(y_true == 1)
    if n_events == 0: nri_events = 0
    else: nri_events = (up_events - down_events) / n_events

    # Non-events
    down_nonevents = np.sum(p_new[y_true == 0] < p_ref[y_true == 0])
    up_nonevents = np.sum(p_new[y_true == 0] > p_ref[y_true == 0])
    n_nonevents = np.sum(y_true == 0)
    if n_nonevents == 0: nri_nonevents = 0
    else: nri_nonevents = (down_nonevents - up_nonevents) / n_nonevents

    nri = nri_events + nri_nonevents
    return nri, idi

def get_bootstrap_ci(y_true, p_ref, p_new, n_rounds=1000, alpha=0.05):
    """执行 Bootstrap 获取 95% CI"""
    nri_boot = []
    idi_boot = []
    
    # 原始样本点估计
    nri_point, idi_point = calculate_nri_idi_point_estimate(y_true, p_ref, p_new)
    
    # Bootstrap 循环
    for i in range(n_rounds):
        # 带放回重采样索引
        indices = resample(np.arange(len(y_true)), replace=True, random_state=i)
        y_sample = y_true[indices]
        
        # 如果重采样样本中只有一类数据，跳过
        if len(np.unique(y_sample)) < 2:
            continue
            
        p_ref_sample = p_ref[indices]
        p_new_sample = p_new[indices]
        
        n, i_val = calculate_nri_idi_point_estimate(y_sample, p_ref_sample, p_new_sample)
        nri_boot.append(n)
        idi_boot.append(i_val)
    
    # 计算百分位数 CI
    lower_p = (alpha / 2) * 100
    upper_p = (1 - alpha / 2) * 100
    
    nri_ci = np.percentile(nri_boot, [lower_p, upper_p])
    idi_ci = np.percentile(idi_boot, [lower_p, upper_p])
    
    # 计算 Z-test P值 (基于 Bootstrap 的标准误 SE)
    nri_se = np.std(nri_boot, ddof=1)
    idi_se = np.std(idi_boot, ddof=1)
    
    z_nri = nri_point / (nri_se + 1e-9)
    p_nri = 2 * (1 - norm.cdf(abs(z_nri)))
    
    z_idi = idi_point / (idi_se + 1e-9)
    p_idi = 2 * (1 - norm.cdf(abs(z_idi)))
    
    return {
        'nri_point': nri_point, 'nri_ci': nri_ci, 'nri_p': p_nri,
        'idi_point': idi_point, 'idi_ci': idi_ci, 'idi_p': p_idi
    }

def run_analysis_and_save_excel(pipeline, filename=OUTPUT_FILE):
    print("\n" + "="*90)
    print(f"🚀 Advanced Statistical Comparison: NRI & IDI with Bootstrap {BOOTSTRAP_ROUNDS}x")
    print("="*90)
    
    # 1. 准备数据与特征
    y_tr = pipeline.train_data['Target'].values
    y_va = pipeline.val_data['Target'].values
    clin = pipeline.selected_clin_vars
    
    feature_sets = {
        'Model 1': clin,
        'Model 2': clin + ['Rad_score'],
        'Model 4': clin + ['Rad_score', 'DL_score', 'Hab_score']
    }
    
    # 2. 计算验证集概率
    probs_val = {}
    print("🤖 Re-fitting Logistic Regression models to generate validation probabilities...")
    for name, feats in feature_sets.items():
        X_tr = pipeline.train_data[feats].apply(pd.to_numeric, errors='coerce').values
        X_va = pipeline.val_data[feats].apply(pd.to_numeric, errors='coerce').values
        
        imp = SimpleImputer(strategy='mean')
        X_tr = imp.fit_transform(X_tr)
        X_va = imp.transform(X_va)
        
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_va = sc.transform(X_va)
        
        lr = LogisticRegression(random_state=SEED, max_iter=5000)
        lr.fit(X_tr, y_tr)
        probs_val[name] = lr.predict_proba(X_va)[:, 1]

    # 3. 定义对比
    comparisons = [
        ("Model 1", "Model 2", "Clin vs Clin+Rad"),
        ("Model 2", "Model 4", "Clin+Rad vs Full Fusion"),
        ("Model 1", "Model 4", "Clin vs Full Fusion")
    ]
    
    # 4. 执行分析并收集数据
    table_data = []
    
    # 格式化表头输出
    header = f"{'Comparison':<25} | {'NRI (95% CI)':<30} | {'P-val':<10} | {'IDI (95% CI)':<30} | {'P-val':<10}"
    print("-" * len(header))
    print(header)
    print("-" * len(header))
    
    for ref_name, new_name, desc in comparisons:
        p_ref = probs_val[ref_name]
        p_new = probs_val[new_name]
        
        res = get_bootstrap_ci(y_va, p_ref, p_new, n_rounds=BOOTSTRAP_ROUNDS)
        
        # 格式化输出字符串
        nri_str = f"{res['nri_point']:.3f} ({res['nri_ci'][0]:.3f}–{res['nri_ci'][1]:.3f})"
        idi_str = f"{res['idi_point']:.3f} ({res['idi_ci'][0]:.3f}–{res['idi_ci'][1]:.3f})"
        
        print(f"{desc:<25} | {nri_str:<30} | {res['nri_p']:.4f}     | {idi_str:<30} | {res['idi_p']:.4f}")
        
        # 收集每一行数据用于 Excel
        row = {
            "Comparison": desc,
            "NRI_Formatted": nri_str, # 方便直接复制到 Word 表格
            "NRI_P_Value": res['nri_p'],
            "IDI_Formatted": idi_str, # 方便直接复制到 Word 表格
            "IDI_P_Value": res['idi_p'],
            # 以下为原始数值，方便绘图或再处理
            "NRI_Point": res['nri_point'],
            "NRI_95_CI_Lower": res['nri_ci'][0],
            "NRI_95_CI_Upper": res['nri_ci'][1],
            "IDI_Point": res['idi_point'],
            "IDI_95_CI_Lower": res['idi_ci'][0],
            "IDI_95_CI_Upper": res['idi_ci'][1]
        }
        table_data.append(row)

    print("-" * len(header))
    
    # 5. 保存到 Excel
    df_results = pd.DataFrame(table_data)
    
    # 调整列顺序，让格式化好的数据排在前面
    cols_order = [
        "Comparison", 
        "NRI_Formatted", "NRI_P_Value", 
        "IDI_Formatted", "IDI_P_Value",
        "NRI_Point", "NRI_95_CI_Lower", "NRI_95_CI_Upper",
        "IDI_Point", "IDI_95_CI_Lower", "IDI_95_CI_Upper"
    ]
    df_results = df_results[cols_order]
    
    df_results.to_excel(filename, index=False)
    print(f"\n✅ Results successfully saved to Excel file: {filename}")
    return df_results

# --- 执行 ---
if 'pipe' in locals() and pipe.train_data is not None:
    _ = run_analysis_and_save_excel(pipe)
else:
    print("❌ 请先运行主 Pipeline 代码块，确保 'pipe' 对象已生成。")

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import norm

# =========================
# 配置
# =========================
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

OUT_XLSX = os.path.join(BASE_DIR, "AUC_Compare_Val_AllModels_DeLong.xlsx")

N_BOOT = 5000
SEED = 42
ALPHA = 0.05
USE_DELONG = True

# 你当前“最终模型”用来做 bootstrap 对比（可按需改）
REF_MODEL = "Model 4 (+Hab)"

assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"

# =========================
# 自动收集所有模型（包含 Model1）
# 优先使用 models.keys() 的顺序，否则用 pred_store 里可用项
# =========================
def _has_val_preds(d):
    return isinstance(d, dict) and ("y_val" in d) and ("p_val" in d)

if "models" in globals() and hasattr(models, "keys"):
    model_list = [k for k in models.keys() if k in pred_store and _has_val_preds(pred_store[k])]
else:
    model_list = [k for k in pred_store.keys() if _has_val_preds(pred_store[k])]

assert len(model_list) >= 2, "可用于验证集比较的模型数量不足（至少需要2个）。"

# =========================
# 读取 y_val / p_val，并检查所有模型的 y_val 完全一致
# =========================
y_ref = np.asarray(pred_store[model_list[0]]["y_val"]).astype(int)
n = len(y_ref)

p_dict = {}
for name in model_list:
    y_i = np.asarray(pred_store[name]["y_val"]).astype(int)
    assert len(y_i) == n, f"{name} 的 y_val 长度({len(y_i)})与其它模型不一致({n})"
    assert np.array_equal(y_i, y_ref), f"{name} 的 y_val 与其它模型不一致（DeLong 要求同一批样本）"
    p_dict[name] = np.asarray(pred_store[name]["p_val"]).astype(float)

y = y_ref

# =========================
# 工具函数
# =========================
def safe_auc(y_true, p_pred):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, p_pred)

def ci_percentile(arr, alpha=0.05):
    lo = np.nanpercentile(arr, 100 * (alpha / 2))
    hi = np.nanpercentile(arr, 100 * (1 - alpha / 2))
    return float(lo), float(hi)

def p_two_sided_from_boot(delta_arr):
    # 双侧：检验 delta=0
    p_neg = np.mean(delta_arr <= 0)
    p_pos = np.mean(delta_arr >= 0)
    return float(2 * min(p_neg, p_pos))

def p_adjust_bh(pvals):
    """Benjamini–Hochberg FDR"""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out = np.empty_like(q)
    out[order] = q
    return out

# =========================
# AUC 汇总（原始）
# =========================
auc_table = pd.DataFrame(
    [{"Model": name, "AUC_Val": safe_auc(y, p_dict[name])} for name in model_list]
).sort_values("AUC_Val", ascending=False).reset_index(drop=True)

# =========================
# Bootstrap：REF_MODEL vs 其它所有模型（配对重采样）
# =========================
rng = np.random.default_rng(SEED)

assert REF_MODEL in model_list, f"REF_MODEL={REF_MODEL} 不在当前可用模型列表中：{model_list}"
p_ref = p_dict[REF_MODEL]
auc_ref = safe_auc(y, p_ref)

boot_rows = []
for other in model_list:
    if other == REF_MODEL:
        continue

    p_other = p_dict[other]
    auc_other = safe_auc(y, p_other)
    d_ref_other = auc_ref - auc_other

    boot_auc_ref, boot_auc_other, boot_delta = [], [], []

    for _ in range(N_BOOT):
        idx = rng.integers(0, n, size=n)  # 有放回
        y_b = y[idx]
        if len(np.unique(y_b)) < 2:
            continue

        a_ref = roc_auc_score(y_b, p_ref[idx])
        a_oth = roc_auc_score(y_b, p_other[idx])

        boot_auc_ref.append(a_ref)
        boot_auc_other.append(a_oth)
        boot_delta.append(a_ref - a_oth)

    boot_auc_ref = np.asarray(boot_auc_ref)
    boot_auc_other = np.asarray(boot_auc_other)
    boot_delta = np.asarray(boot_delta)

    ref_ci = ci_percentile(boot_auc_ref, ALPHA)
    oth_ci = ci_percentile(boot_auc_other, ALPHA)
    d_ci = ci_percentile(boot_delta, ALPHA)
    p_boot = p_two_sided_from_boot(boot_delta)

    boot_rows.append({
        "Comparison": f"{REF_MODEL} vs {other}",
        "AUC_REF": auc_ref, "AUC_REF_95%CI": f"[{ref_ci[0]:.3f}, {ref_ci[1]:.3f}]",
        "AUC_Other": auc_other, "AUC_Other_95%CI": f"[{oth_ci[0]:.3f}, {oth_ci[1]:.3f}]",
        "Delta(AUC_REF-Other)": d_ref_other,
        "Delta_95%CI": f"[{d_ci[0]:.3f}, {d_ci[1]:.3f}]",
        "P_boot(two-sided)": p_boot
    })

boot_summary = pd.DataFrame(boot_rows)

# =========================
# DeLong：所有模型两两比较（配对/相关ROC）
# =========================
delong_allpairs = pd.DataFrame()

if USE_DELONG:
    def _compute_midrank(x):
        J = np.argsort(x)
        Z = x[J]
        N = len(x)
        T = np.zeros(N, dtype=float)
        i = 0
        while i < N:
            j = i
            while j < N and Z[j] == Z[i]:
                j += 1
            T[i:j] = 0.5 * (i + j - 1) + 1
            i = j
        T2 = np.empty(N, dtype=float)
        T2[J] = T
        return T2

    def _fast_delong(predictions_sorted_transposed, label_1_count):
        m = label_1_count
        n0 = predictions_sorted_transposed.shape[1] - m
        pos = predictions_sorted_transposed[:, :m]
        neg = predictions_sorted_transposed[:, m:]
        k = predictions_sorted_transposed.shape[0]

        tx = np.empty((k, m), dtype=float)
        ty = np.empty((k, n0), dtype=float)
        tz = np.empty((k, m + n0), dtype=float)

        for r in range(k):
            tx[r, :] = _compute_midrank(pos[r, :])
            ty[r, :] = _compute_midrank(neg[r, :])
            tz[r, :] = _compute_midrank(predictions_sorted_transposed[r, :])

        aucs = (tz[:, :m].sum(axis=1) - m * (m + 1) / 2) / (m * n0)
        v01 = (tz[:, :m] - tx) / n0
        v10 = 1 - (tz[:, m:] - ty) / m

        sx = np.cov(v01)
        sy = np.cov(v10)
        s = sx / m + sy / n0
        return aucs, s

    def delong_2sample(y_true, p_a, p_b):
        # 返回：auc_a, auc_b, delta(b-a), se, z, p, ci
        y_true = np.asarray(y_true).astype(int)
        p_a = np.asarray(p_a).astype(float)
        p_b = np.asarray(p_b).astype(float)

        order = np.argsort(-y_true)  # positives first
        y_sorted = y_true[order]
        p_a_sorted = p_a[order]
        p_b_sorted = p_b[order]

        m = int(y_sorted.sum())
        preds = np.vstack([p_a_sorted, p_b_sorted])
        aucs, cov = _fast_delong(preds, m)

        auc_a, auc_b = float(aucs[0]), float(aucs[1])
        delta = auc_b - auc_a
        var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
        se = float(np.sqrt(max(var, 1e-12)))
        z = float(delta / se)
        p = float(2 * (1 - norm.cdf(abs(z))))
        ci = (delta - 1.96 * se, delta + 1.96 * se)
        return auc_a, auc_b, delta, se, z, p, ci

    rows = []
    for i in range(len(model_list)):
        for j in range(i + 1, len(model_list)):
            a = model_list[i]
            b = model_list[j]
            auc_a, auc_b, delta, se, z, p, ci = delong_2sample(y, p_dict[a], p_dict[b])
            rows.append({
                "Model_A": a,
                "Model_B": b,
                "AUC_A": auc_a,
                "AUC_B": auc_b,
                "Delta(B-A)": delta,
                "SE": se,
                "Z": z,
                "P_DeLong(two-sided)": p,
                "Delta_95%CI(approx)": f"[{ci[0]:.3f}, {ci[1]:.3f}]"
            })

    delong_allpairs = pd.DataFrame(rows)

    # 多重比较校正（可选但推荐：两两比较很多时）
    if len(delong_allpairs) > 0:
        pvals = delong_allpairs["P_DeLong(two-sided)"].values
        delong_allpairs["P_Bonferroni"] = np.minimum(pvals * len(pvals), 1.0)
        delong_allpairs["P_BH_FDR"] = p_adjust_bh(pvals)

# =========================
# 导出 Excel
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    auc_table.to_excel(w, index=False, sheet_name="AUC_Val_Summary")
    boot_summary.to_excel(w, index=False, sheet_name="Bootstrap_REF_vs_All")
    if USE_DELONG:
        delong_allpairs.to_excel(w, index=False, sheet_name="DeLong_AllPairs")
    pd.DataFrame({
        "Notes":[
            f"AUC_Summary: AUC on the same validation set for all models (n={n}).",
            f"Bootstrap_REF_vs_All: paired bootstrap on Val set, ref={REF_MODEL}, n_boot={N_BOOT}, two-sided p via sign proportion.",
            "DeLong_AllPairs: correlated ROC AUC comparison for all model pairs; approx CI via normal.",
            "Multiple testing: provided Bonferroni and BH-FDR adjusted p-values in DeLong_AllPairs (optional).",
            "Interpretation: Delta_95%CI includes 0 or p>=0.05 -> not statistically significant."
        ]
    }).to_excel(w, index=False, sheet_name="Notes")

print("Saved:", OUT_XLSX)

display(auc_table)
display(boot_summary)
if USE_DELONG:
    display(delong_allpairs)


## 9. Fixed-threshold performance (Supplementary Figure S2)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

# ========= 配置 =========
# 建议：用训练集 OOF 概率选 Youden 阈值，再应用到 Val（更规范）
# 你如果想只对最终模型做：把 model_names 改成 ["Model 4 (+Hab)"]
model_names = [
    "Model 1 (Clin Only)",
    "Model 2 (+Rad)",
    "Model 3 (+DL)",
    "Model 4 (+Hab)",
]

try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

# ========= 检查 pred_store =========
assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"
for m in model_names:
    assert m in pred_store, f"pred_store 缺少模型：{m}"
    assert "y_train" in pred_store[m], f"{m} 缺少 y_train"
    assert "y_val" in pred_store[m], f"{m} 缺少 y_val"
    # 优先使用 OOF
    if "p_train_oof" not in pred_store[m]:
        raise RuntimeError(f"{m} 缺少 p_train_oof（建议用 OOF 选阈值）。请先生成 OOF 概率。")

def youden_threshold(y_true, p_pred):
    """
    Youden index: J = Sensitivity + Specificity - 1 = TPR - FPR
    返回：最佳阈值、对应的(TPR, FPR, J)
    """
    fpr, tpr, thr = roc_curve(y_true, p_pred)
    J = tpr - fpr
    idx = np.argmax(J)
    return float(thr[idx]), float(tpr[idx]), float(fpr[idx]), float(J[idx])

def metrics_at_threshold(y_true, p_pred, thr):
    y_hat = (p_pred >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan  # Sensitivity / Recall
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan  # Specificity
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else np.nan  # Precision / PPV
    npv  = tn / (tn + fn) if (tn + fn) > 0 else np.nan  # NPV
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan

    return {
        "Threshold": thr,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Sensitivity": sens,
        "Specificity": spec,
        "PPV": ppv,
        "NPV": npv,
        "Accuracy": acc
    }

rows = []
cm_rows = []

for m in model_names:
    y_tr = np.asarray(pred_store[m]["y_train"]).astype(int)
    p_tr = np.asarray(pred_store[m]["p_train_oof"]).astype(float)  # 用 OOF 选阈值

    y_va = np.asarray(pred_store[m]["y_val"]).astype(int)
    p_va = np.asarray(pred_store[m]["p_val"]).astype(float)

    thr, tpr_best, fpr_best, J_best = youden_threshold(y_tr, p_tr)

    auc_tr_oof = roc_auc_score(y_tr, p_tr)
    auc_va = roc_auc_score(y_va, p_va)

    tr_metrics = metrics_at_threshold(y_tr, p_tr, thr)
    va_metrics = metrics_at_threshold(y_va, p_va, thr)

    rows.append({
        "Model": m,
        "Youden threshold (from Train OOF)": thr,
        "Youden J (Train OOF)": J_best,
        "TPR@thr (Train OOF)": tpr_best,
        "FPR@thr (Train OOF)": fpr_best,
        "AUC Train (OOF)": auc_tr_oof,
        "AUC Val": auc_va,
        # 验证集指标（用于论文报告）
        "Sensitivity (Val)": va_metrics["Sensitivity"],
        "Specificity (Val)": va_metrics["Specificity"],
        "PPV (Val)": va_metrics["PPV"],
        "NPV (Val)": va_metrics["NPV"],
        "Accuracy (Val)": va_metrics["Accuracy"],
    })

    cm_rows.append({
        "Model": m,
        "Threshold": thr,
        "TP (Val)": va_metrics["TP"],
        "FP (Val)": va_metrics["FP"],
        "TN (Val)": va_metrics["TN"],
        "FN (Val)": va_metrics["FN"],
    })

df_perf = pd.DataFrame(rows)
df_cm = pd.DataFrame(cm_rows)

# 显示（Jupyter）
display(df_perf)
display(df_cm)

# 可选：导出到results（如果你要做期刊表格素材）
out_xlsx = os.path.join(BASE_DIR, "YoudenThreshold_Val_Performance.xlsx")
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
    df_perf.to_excel(w, index=False, sheet_name="Val_Performance_Youden")
    df_cm.to_excel(w, index=False, sheet_name="Val_ConfusionMatrix")
print("Saved:", out_xlsx)


In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve

# ======================
# 论文级输出参数（不改变颜色）
# ======================
mpl.rcParams.update({
    "pdf.fonttype": 42,     # TrueType 字体嵌入
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

# ======================
# 配置
# ======================
m = "Model 4 (+Hab)"   # 你要画哪个模型就改这里
class_names = ["Non-LNM", "LNM"]

# 输出路径
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

out_png = os.path.join(BASE_DIR, "ConfusionMatrix_Train_vs_Val_Model4.png")
out_pdf = os.path.join(BASE_DIR, "ConfusionMatrix_Train_vs_Val_Model4.pdf")
DPI_PNG = 600

assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"
assert m in pred_store, f"pred_store 中没有 {m}"

y_tr = np.asarray(pred_store[m]["y_train"]).astype(int)
y_va = np.asarray(pred_store[m]["y_val"]).astype(int)

# 训练集概率：优先 fit，否则用 OOF
if "p_train_fit" in pred_store[m]:
    p_tr = np.asarray(pred_store[m]["p_train_fit"]).astype(float)
else:
    p_tr = np.asarray(pred_store[m]["p_train_oof"]).astype(float)

p_va = np.asarray(pred_store[m]["p_val"]).astype(float)

# ======================
# 阈值：默认用训练集(OOF)的 Youden 阈值
# ======================
def youden_threshold(y_true, p_pred):
    fpr, tpr, thr = roc_curve(y_true, p_pred)
    J = tpr - fpr
    idx = np.argmax(J)
    return float(thr[idx])

# 如果想强制指定 cutoff（例如 0.676），用下一行替换：
# cutoff = 0.676
cutoff = youden_threshold(y_tr, np.asarray(pred_store[m]["p_train_oof"]).astype(float))

# ======================
# 画混淆矩阵（计数 + 行内百分比）
# ======================
def plot_cm(ax, y_true, p_pred, cutoff, title, class_names, cmap="Blues", vmax=None):
    y_hat = (p_pred >= cutoff).astype(int)
    cm = confusion_matrix(y_true, y_hat, labels=[0, 1])  # [[TN, FP],[FN, TP]]

    im = ax.imshow(cm, cmap=cmap, vmin=0, vmax=vmax)

    ax.set_title(f"{title}\n(Cutoff={cutoff:.3f})", fontsize=16, fontweight="bold")
    ax.set_xlabel("Predicted Label", fontsize=13)
    ax.set_ylabel("True Label", fontsize=13)

    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(class_names, fontsize=12)
    ax.set_yticklabels(class_names, fontsize=12, rotation=90, va="center")

    # 标注：count + (row %)
    for i in range(2):
        row_sum = cm[i, :].sum()
        for j in range(2):
            cnt = cm[i, j]
            pct = (cnt / row_sum * 100) if row_sum > 0 else 0.0

            # 深色格子用白字（不改配色，只调整文字可读性）
            use_white = (vmax is not None and cnt > 0.5 * vmax)
            ax.text(
                j, i, f"{cnt}\n({pct:.1f}%)",
                ha="center", va="center",
                fontsize=16, fontweight="bold",
                color="white" if use_white else "black"
            )

    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)

    return im, cm

# 两个矩阵统一色阶，便于视觉对比
cm_tr = confusion_matrix(y_tr, (p_tr >= cutoff).astype(int), labels=[0, 1])
cm_va = confusion_matrix(y_va, (p_va >= cutoff).astype(int), labels=[0, 1])
vmax = max(cm_tr.max(), cm_va.max())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
im1, _ = plot_cm(axes[0], y_tr, p_tr, cutoff, "Training Cohort", class_names, vmax=vmax)
im2, _ = plot_cm(axes[1], y_va, p_va, cutoff, "Validation Cohort", class_names, vmax=vmax)

fig.tight_layout()

# 高分辨率PNG + 矢量PDF
fig.savefig(out_png, dpi=DPI_PNG)
fig.savefig(out_pdf)
plt.show()
plt.close(fig)

print("Saved:", out_png)
print("Saved:", out_pdf)
print(f"Cutoff used: {cutoff:.3f}")


In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ========= 论文级输出参数（不改变颜色） =========
mpl.rcParams.update({
    "pdf.fonttype": 42,     # TrueType 字体嵌入
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

# ========== 前置：取数据 ==========
m4 = "Model 4 (+Hab)"
assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"
assert m4 in pred_store, f"pred_store 中没有 {m4}"

y_val = np.asarray(pred_store[m4]["y_val"]).astype(int)
p_val = np.asarray(pred_store[m4]["p_val"]).astype(float)

# 阈值（Train OOF–Youden）
assert "cutoff" in globals(), "未找到 cutoff（Train OOF–Youden），请先运行阈值计算 cell。"
thr = float(cutoff)

# 分组
p_non = p_val[y_val == 0]
p_lnm = p_val[y_val == 1]

# 混淆矩阵（Val）
pred = (p_val >= thr).astype(int)
TN = int(((y_val == 0) & (pred == 0)).sum())
FP = int(((y_val == 0) & (pred == 1)).sum())
FN = int(((y_val == 1) & (pred == 0)).sum())
TP = int(((y_val == 1) & (pred == 1)).sum())

# 输出路径（PNG + PDF）
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

out_png = os.path.join(BASE_DIR, "Val_PredProb_Violin_Model4.png")
out_pdf = os.path.join(BASE_DIR, "Val_PredProb_Violin_Model4.pdf")
DPI_PNG = 600

# ========== 画图 ==========
fig, ax = plt.subplots(figsize=(7.6, 6.4))

data = [p_non, p_lnm]
positions = [1, 2]

# --- 小提琴：浅色、半透明（颜色保持不变） ---
parts = ax.violinplot(
    data,
    positions=positions,
    widths=0.8,
    showmeans=False,
    showmedians=False,
    showextrema=False
)
for body in parts["bodies"]:
    body.set_facecolor("lightgray")
    body.set_edgecolor("black")
    body.set_alpha(0.35)
    body.set_linewidth(1.2)

# --- 箱线：深色、显眼（颜色保持不变） ---
ax.boxplot(
    data,
    positions=positions,
    widths=0.28,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="dimgray", edgecolor="black", linewidth=1.6, alpha=0.55),
    medianprops=dict(color="black", linewidth=2.0),
    whiskerprops=dict(color="black", linewidth=1.4),
    capprops=dict(color="black", linewidth=1.4)
)

# --- 散点：深灰、半透明（不抢）---
rng = np.random.default_rng(0)
for i, vals in enumerate(data):
    x = np.full_like(vals, positions[i], dtype=float)
    x = x + rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(x, vals, s=18, c="black", alpha=0.35, linewidths=0)

# --- 阈值线：深色虚线（颜色保持不变） ---
ax.axhline(thr, linestyle="--", linewidth=1.5, color="black")

# 轴与标题
ax.set_xticks(positions)
ax.set_xticklabels(["Non-LNM", "LNM"], fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Predicted probability (Model 4)", fontsize=12)
ax.set_title("Validation Predicted Probability Distribution (Model 4)", fontsize=14)

# 右下角注释
txt = (
    f"Cutoff (Train OOF–Youden) = {thr:.3f}\n"
    f"Val: TN={TN}, FP={FP}, FN={FN}, TP={TP}"
)
ax.text(
    0.98, 0.06, txt,
    transform=ax.transAxes,
    ha="right", va="bottom",
    fontsize=10, color="black"
)

fig.tight_layout()
fig.savefig(out_png, dpi=DPI_PNG)
fig.savefig(out_pdf)
plt.show()
plt.close(fig)

print("Saved:", out_png)
print("Saved:", out_pdf)


## 10. Calibration metrics and locked Model 4 coefficients

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression

# ===== 配置 =====
model_names = [
    "Model 1 (Clin Only)",
    "Model 2 (+Rad)",
    "Model 3 (+DL)",
    "Model 4 (+Hab)"
]

try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

out_xlsx = os.path.join(BASE_DIR, "Val_Calibration_Metrics_Brier_Slope_Intercept.xlsx")

assert "pred_store" in globals(), "未找到 pred_store，请先运行生成预测的 cell。"
for m in model_names:
    assert m in pred_store, f"pred_store 缺少模型：{m}"
    assert "y_val" in pred_store[m] and "p_val" in pred_store[m], f"{m} 缺少 y_val 或 p_val"

def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

rows = []
for m in model_names:
    y = np.asarray(pred_store[m]["y_val"]).astype(int)
    p = np.asarray(pred_store[m]["p_val"]).astype(float)

    # AUC（可选一起输出，便于对照）
    auc = roc_auc_score(y, p)

    # Brier score（越小越好）
    brier = brier_score_loss(y, p)

    # Calibration intercept & slope（logistic calibration regression）
    # 设 X=logit(p)，拟合 y ~ a + b*X，得到 intercept=a, slope=b
    X = logit(p).reshape(-1, 1)
    cal_lr = LogisticRegression(
        penalty="l2", C=1e6, solver="lbfgs", max_iter=10000
    )
    cal_lr.fit(X, y)
    intercept = float(cal_lr.intercept_[0])
    slope = float(cal_lr.coef_[0][0])

    rows.append({
        "Model": m,
        "AUC (Val)": auc,
        "Brier score (Val)": brier,
        "Calibration intercept (Val)": intercept,
        "Calibration slope (Val)": slope
    })

df_cal = pd.DataFrame(rows)

# 统一显示格式（可选）
df_show = df_cal.copy()
for c in ["AUC (Val)", "Brier score (Val)", "Calibration intercept (Val)", "Calibration slope (Val)"]:
    df_show[c] = df_show[c].map(lambda x: f"{x:.3f}")

display(df_show)

# 导出
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
    df_cal.to_excel(w, index=False, sheet_name="Val_Calibration_Metrics")
    pd.DataFrame({
        "Notes": [
            "Brier score: lower is better.",
            "Calibration intercept close to 0 and slope close to 1 indicate good calibration.",
            "Intercept/slope estimated by logistic calibration regression: y ~ a + b*logit(p)."
        ]
    }).to_excel(w, index=False, sheet_name="Notes")

print("Saved:", out_xlsx)


In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

# ============================================================
#  Model 4 fusion LR OR table (Train set)
#  - Continuous vars: Z-score, OR per 1 SD increase
#  - Stage: categorical T1/T2/T3, dummy-coded with T1 as reference
#  - Wald 95% CI from statsmodels Logit
#  - Robust dtype handling to avoid "object dtype" errors
#  - Export to results Excel
# ============================================================

# ===== Path =====
try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

out_xlsx = os.path.join(BASE_DIR, "Table_Model4_Fusion_LR_OR_Train.xlsx")

# ===== Require pipeline object =====
assert "pipe" in globals(), (
    "未找到 pipe。请先运行你的主 pipeline（至少跑到 build_signatures，"
    "确保 pipe.train_data 中已有 Rad_score/DL_score/Hab_score）。"
)

df = pipe.train_data.copy()
assert "Target" in df.columns, "pipe.train_data 中缺少 Target"

# ===== Variables (as you specified) =====
stage_col = "分期(1T12T23T3)"
feat_cols = ["c_Tumor_Size", stage_col, "Rad_score", "DL_score", "Hab_score"]

missing = [c for c in feat_cols if c not in df.columns]
assert not missing, f"train_data 缺少这些列：{missing}"

# ===== y =====
y = df["Target"].astype(int).values

# ===== X raw =====
X_raw = df[feat_cols].copy()

# --------- Missing handling ----------
# numeric: mean impute
# stage: mode impute (after mapping)
for c in X_raw.columns:
    if c == stage_col:
        X_raw[c] = X_raw[c].astype(str).str.strip()
    else:
        X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")
        X_raw[c] = X_raw[c].fillna(X_raw[c].mean())

# --------- Stage mapping (robust) ----------
# Compatible with: 1/2/3, "1"/"2"/"3", "T1"/"T2"/"T3", "1T1"/"2T2"/"3T3", etc.
s = X_raw[stage_col].astype(str).str.strip().str.upper()

stage_map = {
    "1": "T1", "2": "T2", "3": "T3",
    "T1": "T1", "T2": "T2", "T3": "T3",
    "1T1": "T1", "2T2": "T2", "3T3": "T3",
}

s2 = s.replace(stage_map)

# If still not in {T1,T2,T3}, set NaN then fill by mode
s2 = s2.where(s2.isin(["T1", "T2", "T3"]), np.nan)
mode_stage = s2.mode(dropna=True)
fill_stage = mode_stage.iloc[0] if len(mode_stage) else "T1"
X_raw[stage_col] = s2.fillna(fill_stage)

# force categorical with fixed levels
X_raw[stage_col] = pd.Categorical(X_raw[stage_col], categories=["T1", "T2", "T3"], ordered=False)

# --------- One-hot encode stage (T1 reference) ----------
X = pd.get_dummies(X_raw, columns=[stage_col], drop_first=True)

# Dummy columns created for stage
dummy_cols = [c for c in X.columns if c.startswith(stage_col + "_")]

# --------- Z-score numeric (not dummies) ----------
num_cols = [c for c in X.columns if c not in dummy_cols]

means = X[num_cols].mean()
stds = X[num_cols].std(ddof=0)

# Drop zero-variance numeric columns to avoid NaN after z-score
zero_var = stds[stds == 0].index.tolist()
if len(zero_var) > 0:
    X = X.drop(columns=zero_var)
    num_cols = [c for c in num_cols if c not in zero_var]
    means = X[num_cols].mean()
    stds = X[num_cols].std(ddof=0)

X[num_cols] = (X[num_cols] - means) / stds

# --------- Critical: enforce numeric dtype for statsmodels ----------
X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

# Add constant and enforce float
X_sm = sm.add_constant(X, has_constant="add").astype(float)

# --------- Sanity check: stage dummies not all-zero ----------
# If both are 0, you likely mapped everything to T1; fix mapping/values.
if len(dummy_cols) > 0:
    print("Stage dummy sums:\n", X_sm[dummy_cols].sum())

# --------- Fit Logit ----------
model = sm.Logit(y.astype(float), X_sm)

# If you still run into convergence issues, you can increase maxiter.
res = model.fit(disp=False, method="lbfgs", maxiter=500)

# --------- Extract OR / CI / P ----------
params = res.params
conf = res.conf_int()
pvals = res.pvalues

tbl = pd.DataFrame({
    "Term": params.index,
    "Beta (log-odds)": params.values,
    "OR": np.exp(params.values),
    "OR 95%CI (lower)": np.exp(conf[0].values),
    "OR 95%CI (upper)": np.exp(conf[1].values),
    "P value": pvals.values
})

# --------- Interpretation text ----------
interpret = []
for term in tbl["Term"]:
    if term == "const":
        interpret.append("Intercept")
    elif term in num_cols:
        interpret.append("Per 1 SD increase")
    elif term in dummy_cols:
        # term like: 分期(1T12T23T3)_T2
        lvl = term.split("_")[-1]
        interpret.append(f"{lvl} vs T1")
    else:
        interpret.append("")
tbl["Interpretation"] = interpret

# --------- Display-friendly formatting (keeps raw numeric in export) ----------
tbl_disp = tbl.copy()
tbl_disp["Beta (log-odds)"] = tbl_disp["Beta (log-odds)"].map(lambda x: f"{x:.3f}")
tbl_disp["OR"] = tbl_disp["OR"].map(lambda x: f"{x:.3f}")
tbl_disp["OR 95%CI (lower)"] = tbl_disp["OR 95%CI (lower)"].map(lambda x: f"{x:.3f}")
tbl_disp["OR 95%CI (upper)"] = tbl_disp["OR 95%CI (upper)"].map(lambda x: f"{x:.3f}")
tbl_disp["P value"] = tbl_disp["P value"].map(lambda x: f"{x:.4f}" if x >= 1e-4 else "<0.0001")

display(tbl_disp)

# --------- Export Excel ----------
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
    tbl.to_excel(w, index=False, sheet_name="Model4_OR_WaldCI")
    pd.DataFrame({
        "Standardized_numeric_terms (per 1 SD increase)": num_cols,
        "Mean_used_for_zscore": [means.get(c, np.nan) for c in num_cols],
        "SD_used_for_zscore": [stds.get(c, np.nan) for c in num_cols],
    }).to_excel(w, index=False, sheet_name="Zscore_Info")
    pd.DataFrame({
        "Notes": [
            "Training set only.",
            "Stage treated as categorical; reference=T1; dummy terms are T2 vs T1 and T3 vs T1.",
            "Numeric predictors z-scored; OR interpretable per 1 SD increase.",
            "95% CI are Wald intervals from statsmodels Logit."
        ]
    }).to_excel(w, index=False, sheet_name="Notes")

print("Saved:", out_xlsx)
print(res.summary())


## 11. Prespecified sensitivity analysis (Table S3D)

In [ ]:
# Sensitivity analysis (Val): remove c_Tumor_Size from Model 4 (keep T stage + Rad/DL/Hab)
import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

# ===== prerequisites =====
assert "pipe" in globals(), "请先运行主 pipeline，确保 pipe.train_data / pipe.val_data 已存在。"
train_df = pipe.train_data.copy()
val_df   = pipe.val_data.copy()

try:
    BASE_DIR = BASE_DIR
except NameError:
    BASE_DIR = str(OUTPUT_DIR)
os.makedirs(BASE_DIR, exist_ok=True)

try:
    SEED = SEED
except NameError:
    SEED = 42

out_xlsx = os.path.join(BASE_DIR, "Sensitivity_Remove_TumorSize_Val.xlsx")

# ===== features (方案1：留分期、删肿瘤大小) =====
stage_col = "分期(1T12T23T3)"
feats_full   = [stage_col, "Rad_score", "DL_score", "Hab_score", "c_Tumor_Size"]  # 原 Model4
feats_nosize = [stage_col, "Rad_score", "DL_score", "Hab_score"]                 # 删 size

for c in ["Target"] + feats_full:
    assert c in train_df.columns, f"Train 缺少列：{c}"
    assert c in val_df.columns,   f"Val 缺少列：{c}"

y_tr = train_df["Target"].astype(int).values
y_va = val_df["Target"].astype(int).values

def _prep_X(df_tr, df_va, feats):
    Xtr = df_tr[feats].copy()
    Xva = df_va[feats].copy()

    # one-hot for any non-numeric (robust to categorical stage)
    Xtr = pd.get_dummies(Xtr, drop_first=True)
    Xva = pd.get_dummies(Xva, drop_first=True)
    Xva = Xva.reindex(columns=Xtr.columns, fill_value=0)

    # numeric coercion
    Xtr = Xtr.apply(pd.to_numeric, errors="coerce")
    Xva = Xva.apply(pd.to_numeric, errors="coerce")

    imp = SimpleImputer(strategy="mean")
    Xtr_imp = imp.fit_transform(Xtr)
    Xva_imp = imp.transform(Xva)

    sc = StandardScaler()
    Xtr_sc = sc.fit_transform(Xtr_imp)
    Xva_sc = sc.transform(Xva_imp)

    return Xtr_sc, Xva_sc, Xtr.columns.tolist()

def fit_predict_metrics(feats):
    Xtr_sc, Xva_sc, feat_names = _prep_X(train_df, val_df, feats)

    lr = LogisticRegression(random_state=SEED, max_iter=5000)
    lr.fit(Xtr_sc, y_tr)

    p_va = lr.predict_proba(Xva_sc)[:, 1]
    auc  = roc_auc_score(y_va, p_va)
    brier = brier_score_loss(y_va, p_va)

    return p_va, auc, brier, feat_names

# ===== fit/eval =====
p_full, auc_full, brier_full, names_full = fit_predict_metrics(feats_full)
p_nosz, auc_nosz, brier_nosz, names_nosz = fit_predict_metrics(feats_nosize)

delta_auc   = auc_nosz - auc_full
delta_brier = brier_nosz - brier_full

# ===== bootstrap CI on Val for deltas (fixed models) =====
rng = np.random.default_rng(SEED)
n_boot = 2000
dA, dB = [], []

idx = np.arange(len(y_va))
for _ in range(n_boot):
    bs = rng.choice(idx, size=len(idx), replace=True)

    # AUC delta
    try:
        a_full = roc_auc_score(y_va[bs], p_full[bs])
        a_nosz = roc_auc_score(y_va[bs], p_nosz[bs])
        dA.append(a_nosz - a_full)
    except ValueError:
        # rare: bootstrap sample contains a single class
        continue

    # Brier delta
    b_full = brier_score_loss(y_va[bs], p_full[bs])
    b_nosz = brier_score_loss(y_va[bs], p_nosz[bs])
    dB.append(b_nosz - b_full)

dA = np.array(dA, dtype=float)
dB = np.array(dB, dtype=float)

def ci95(x):
    return (float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5)))

dA_lo, dA_hi = ci95(dA)
dB_lo, dB_hi = ci95(dB)

# ===== output table =====
rows = [
    {
        "Model (Val)": "Model 4 (+Hab)  (with Tumor size)",
        "Features (expanded)": ", ".join(names_full),
        "AUC (Val)": auc_full,
        "Brier score (Val)": brier_full,
    },
    {
        "Model (Val)": "Model 4 (+Hab)  (without Tumor size)",
        "Features (expanded)": ", ".join(names_nosz),
        "AUC (Val)": auc_nosz,
        "Brier score (Val)": brier_nosz,
    },
    {
        "Model (Val)": "Δ (without − with)",
        "Features (expanded)": "",
        "AUC (Val)": delta_auc,
        "Brier score (Val)": delta_brier,
        "ΔAUC 95%CI (bootstrap)": f"[{dA_lo:.3f}, {dA_hi:.3f}]",
        "ΔBrier 95%CI (bootstrap)": f"[{dB_lo:.3f}, {dB_hi:.3f}]",
    }
]
out_df = pd.DataFrame(rows)

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
    out_df.to_excel(w, index=False, sheet_name="Sensitivity_Val")

print("Saved:", out_xlsx)
print(f"AUC Val: with size = {auc_full:.3f} | without size = {auc_nosz:.3f} | ΔAUC = {delta_auc:+.3f} (95%CI {dA_lo:+.3f} to {dA_hi:+.3f})")
print(f"Brier Val: with size = {brier_full:.3f} | without size = {brier_nosz:.3f} | ΔBrier = {delta_brier:+.3f} (95%CI {dB_lo:+.3f} to {dB_hi:+.3f})")

# ===== sentence-ready output (for your Results text) =====
print(
    "敏感性分析显示，在验证集中移除肿瘤大小后模型区分度与校准性能未见实质变化"
    f"（ΔAUC={delta_auc:+.3f}，95%CI [{dA_lo:+.3f}, {dA_hi:+.3f}]；"
    f"ΔBrier={delta_brier:+.3f}，95%CI [{dB_lo:+.3f}, {dB_hi:+.3f}]），"
    "支持采用更简洁的变量组合。"
)


## 12. SHAP interpretation in the internal test cohort (Figure 6)

In [ ]:
import os
import shap
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def plot_figure_5ab_shap_sci(pipe, models):
    print("\n🎨 正在绘制 Figure 6 A/B: 基于独立验证集的 SHAP 分析...")
    
    save_path = BASE_DIR  
    model_name = 'Model 4 (+Hab)'
    feats = models[model_name]
    
    # --- 1. 数据切分 (Train 用于拟合, Val 用于解释) ---
    X_raw_tr = _coerce_numeric(pipe.train_data, feats)
    y_train = pipe.train_data['Target'].values.astype(int)
    
    X_raw_va = _coerce_numeric(pipe.val_data, feats)
    
    # SCI 级别特征映射
    rename_dict = {
        '分期(1T12T23T3)': 'T stage',
        'c_Tumor_Size': 'Tumor size',
        'Rad_score': 'Rad-score',
        'DL_score': 'DL-score',
        'Hab_score': 'Hab-score'
    }
    clean_feature_names = [rename_dict.get(f, f) for f in feats]
    
    # --- 2. 模型拟合 (严格限定在训练集) ---
    imp = SimpleImputer(strategy='mean')
    X_tr_imp = imp.fit_transform(X_raw_tr.values)
    X_va_imp = imp.transform(X_raw_va.values)
    
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr_imp)
    X_va_sc = sc.transform(X_va_imp)
    
    lr_final = LogisticRegression(random_state=42, max_iter=5000)
    lr_final.fit(X_tr_sc, y_train)
    
    # --- 3. 计算独立验证集的 SHAP ---
    explainer = shap.LinearExplainer(lr_final, X_tr_sc)
    shap_values_va = explainer.shap_values(X_va_sc)
    
    # --- 4. 绘制 Figure 5A: 柱状图 (Bar Plot - 宏观重要性) ---
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values_va, 
        features=X_raw_va, 
        feature_names=clean_feature_names, 
        plot_type="bar", 
        show=False, 
        color="#1f77b4"
    )
    
    ax1 = plt.gca()
    # 强制修改统计学横轴，不添加 title
    ax1.set_xlabel("Mean |SHAP value|", fontsize=12, fontweight='bold')
    # 相对坐标轴定位左上角 A 标
    ax1.text(-0.35, 1.05, 'A', transform=ax1.transAxes, fontsize=22, fontweight='bold', va='top')
    
    for fmt in ['png', 'pdf']:
        fname = os.path.join(save_path, f"Figure_6A_SHAP_Bar.{fmt}")
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✅ 已保存: {fname}")
    plt.close()

    # --- 5. 绘制 Figure 5B: 蜂群图 (Summary Plot - 微观方向性) ---
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values_va, 
        features=X_raw_va, 
        feature_names=clean_feature_names, 
        show=False, 
        cmap="coolwarm"
    )
    
    ax2 = plt.gca()
    # 强制修改统计学横轴，不添加 title
    ax2.set_xlabel("SHAP value (log-odds of LNM)", fontsize=12, fontweight='bold')
    # 相对坐标轴定位左上角 B 标
    ax2.text(-0.35, 1.05, 'B', transform=ax2.transAxes, fontsize=22, fontweight='bold', va='top')
    
    for fmt in ['png', 'pdf']:
        fname = os.path.join(save_path, f"Figure_6B_SHAP_Summary.{fmt}")
        plt.savefig(fname, dpi=600, bbox_inches='tight')
        print(f"  ✅ 已保存: {fname}")
    plt.close()

# 运行保存任务
plot_figure_5ab_shap_sci(pipe, models)